In [11]:
# All imports
import os
import sys
sys.path.insert(0, '../../')

# Import Constants
from scripts.constants import ADG_SKYNET_ROOT_ADDR

# Import Agent Codes + Forecaster
import ppo2 as network
from env import ABREnv
from Patch_TST import PatchTST
import torch

# Import Libraries
from ast import literal_eval
import numpy as np
import pandas as pd
from sklearn.metrics import mutual_info_score
from scipy.stats import chi2_contingency # Import chi2_contingency
from natsort import natsorted  # Import natsorted for natural sorting
import re
import datetime # Import datetime module
from collections import defaultdict

import seaborn as sns
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
from plotly.colors import qualitative
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pprint import pprint
import math
import time

# Set device for PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Constants


In [2]:
S_DIM = [7, 8]
A_DIM = 6
ACTOR_LR_RATE = 1e-4
TRAIN_SEQ_LEN = 48  # take as a train batch
TRAIN_EPOCH = 1000
RANDOM_SEED = 42

FORECASTER_PATH = ADG_SKYNET_ROOT_ADDR + '/pensive/pensive-gamma/TST-models/tst3_epoch_1200_loss_0.1210.pt'
n_token = 12           # lookback window size
input_dim = 1         # univariate input
model_dim = 64        # model/embedding dimension
num_heads = 4         # number of heads for multi-head attention
num_layers = 3        # number of transformer layers
output_dim = 4        # output dimension = horizon
FORECASTER_MODEL = PatchTST(n_token, input_dim, model_dim, num_heads, num_layers, output_dim)
if_MLP = False

VIDEO_BIT_RATE = [300., 750., 1200., 1850., 2850., 4300.] # Define VIDEO_BIT_RATE

# Classes

In [12]:
# quantile_manager.py
import pandas as pd


class PSquareQuantileApproximator:
    """
    Implements the extended P² algorithm for dynamic quantile approximation.
    Tracks multiple quantiles using multiple markers.
    """
    def __init__(self, quantiles):
        """
        Initialize the approximator with a list of quantiles.

        :param quantiles: List of quantiles to estimate (e.g., [0.25, 0.5, 0.75])
        """
        self.quantiles = sorted(quantiles)
        self.num_quantiles = len(self.quantiles)
        self.reset()

    def reset(self):
        """Initialize or reset the quantile approximator."""
        self.initial_sample = []
        self.num_markers = 2 * self.num_quantiles + 3  # Number of markers
        self.q = []  # Marker heights
        self.n = []  # Marker positions
        self.n_desired = []  # Desired marker positions
        self.p_inc = []  # Desired position increments

    def fit(self, X):
        """Fit the model to the data."""
        self.reset()
        return self.partial_fit(X)

    def partial_fit(self, X):
        """Incrementally fit the model to the data."""
        for x in X:
            self._partial_fit_single(x)
        return self

    def _partial_fit_single(self, x):
        """Fit the model to a single data point."""
        # Collect initial samples
        if len(self.initial_sample) < self.num_markers:
            self.initial_sample.append(x)
            if len(self.initial_sample) == self.num_markers:
                self.initial_sample.sort()
                self.q = self.initial_sample.copy()
                self.n = list(range(1, self.num_markers + 1))
                self.p_inc = [0] + self.quantiles + [1] + [1 - q for q in reversed(self.quantiles)] + [0]
                total_p_inc = sum(self.p_inc)
                self.n_desired = [1 + (total_p_inc - sum(self.p_inc[:i+1])) * (self.num_markers - 1)
                                  for i in range(self.num_markers)]
        else:
            # Update markers
            if x < self.q[0]:
                self.q[0] = x  # Update minimum
                k = 0
            elif x >= self.q[-1]:
                self.q[-1] = x  # Update maximum
                k = self.num_markers - 2
            else:
                # Find k such that q[k] <= x < q[k+1]
                k = next(i for i in range(self.num_markers - 1) if self.q[i] <= x < self.q[i+1])
            # Increment positions of markers greater than k
            for i in range(k + 1, self.num_markers):
                self.n[i] += 1
            # Update desired positions
            for i in range(self.num_markers):
                self.n_desired[i] += self.p_inc[i]
            # Adjust marker heights
            for i in range(1, self.num_markers - 1):
                d = self.n_desired[i] - self.n[i]
                if (d >= 1 and self.n[i + 1] - self.n[i] > 1) or (d <= -1 and self.n[i - 1] - self.n[i] < -1):
                    d = int(np.sign(d))
                    q_temp = self._parabolic(i, d)
                    if self.q[i - 1] < q_temp < self.q[i + 1]:
                        self.q[i] = q_temp
                    else:
                        self.q[i] = self._linear(i, d)
                    self.n[i] += d

    def _parabolic(self, i, d):
        """Parabolic prediction for marker height adjustment."""
        i = int(i)
        n_i = self.n[i]
        n_im1 = self.n[i - 1]
        n_ip1 = self.n[i + 1]
        q_i = self.q[i]
        q_im1 = self.q[i - 1]
        q_ip1 = self.q[i + 1]
        numerator = d * ((n_i - n_im1 + d) * (q_ip1 - q_i) / (n_ip1 - n_i) +
                         (n_ip1 - n_i - d) * (q_i - q_im1) / (n_i - n_im1))
        denominator = n_ip1 - n_im1
        return q_i + numerator / denominator

    def _linear(self, i, d):
        """Linear prediction for marker height adjustment."""
        i = int(i)
        return self.q[i] + d * (self.q[i + d] - self.q[i]) / (self.n[i + d] - self.n[i])

    def is_initialized(self):
        """Check if the approximator has been initialized with enough data points."""
        return len(self.q) == self.num_markers

    def get_markers(self):
        """Return the current quantile estimates."""
        if self.is_initialized():
            # The quantile estimates are at positions corresponding to the quantiles
            quantile_indices = [1 + i for i in range(self.num_quantiles)]
            quantile_values = [self.q[i] for i in quantile_indices]
            return [self.q[0]] + quantile_values + [self.q[-1]]  # Include min and max
        else:
            # Return the quantiles from the initial sample
            if self.initial_sample:
                quantile_values = [np.percentile(self.initial_sample, q * 100) for q in self.quantiles]
                return [min(self.initial_sample)] + quantile_values + [max(self.initial_sample)]
            else:
                return None

    def get_state(self):
        """Get the internal state for saving or exporting."""
        return {
            'q': self.q,
            'n': self.n,
            'n_desired': self.n_desired,
            'p_inc': self.p_inc,
            'initial_sample': self.initial_sample
        }

    def set_state(self, state):
        """Set the internal state from saved data."""
        self.q = state['q']
        self.n = state['n']
        self.n_desired = state['n_desired']
        self.p_inc = state['p_inc']
        self.initial_sample = state['initial_sample']

    def get_quantile_names(self):
        """Return the names of the quantiles, including 'min' and 'max'."""
        quantile_names = ['min'] + [f'p{int(round(q * 100))}' for q in self.quantiles] + ['max']
        return quantile_names


# gk_quantile_approximator.py
import bisect

class GKQuantileApproximator:
    """
    Implements the Greenwald-Khanna algorithm for quantile estimation.
    """
    def __init__(self, quantiles, epsilon=0.01):
        """
        :param quantiles: List of quantiles to estimate (e.g., [0.25, 0.5, 0.75])
        :param epsilon: Acceptable error in the quantile approximation.
        """
        self.quantiles = sorted(quantiles)
        self.epsilon = epsilon
        self.reset()

    def reset(self):
        """Reset the GK approximator."""
        self.n = 0  # Total number of data points
        self.summary = []  # List of tuples (value, g, delta)

    def partial_fit(self, X):
        """Incrementally fit the model to the data."""
        for x in X:
            self.insert(x)
        return self

    def insert(self, x):
        """Insert a new data point into the summary."""
        self.n += 1
        r = {'value': x, 'g': 1, 'delta': 0}
        if not self.summary:
            self.summary.append(r)
            return

        idx = bisect.bisect_left([s['value'] for s in self.summary], x)
        if idx == 0:
            delta = 0
        elif idx == len(self.summary):
            delta = 0
        else:
            delta = int(2 * self.epsilon * self.n)
        r['delta'] = delta
        self.summary.insert(idx, r)
        self.compress()

    def compress(self):
        """Compress the summary to maintain the error guarantees."""
        i = 0
        while i < len(self.summary) - 1:
            r1 = self.summary[i]
            r2 = self.summary[i + 1]
            if (r1['g'] + r2['g'] + r2['delta']) <= int(2 * self.epsilon * self.n):
                r2['g'] += r1['g']
                self.summary.pop(i)
            else:
                i += 1

    def get_markers(self):
        """Return the quantile estimates."""
        if not self.summary:
            return None
        markers = [self.summary[0]['value']]  # min
        for quantile in self.quantiles:
            rank_min = 0
            desired_rank = quantile * self.n
            for i, s in enumerate(self.summary):
                rank_min += s['g']
                rank_max = rank_min + s['delta']
                if rank_max >= desired_rank + self.epsilon * self.n:
                    markers.append(s['value'])
                    break
        markers.append(self.summary[-1]['value'])  # max
        return markers

    def is_initialized(self):
        """Check if the approximator has enough data."""
        return len(self.summary) > 0

    def get_state(self):
        """Get the internal state for saving or exporting."""
        return {
            'n': self.n,
            'summary': self.summary
        }

    def set_state(self, state):
        """Set the internal state from saved data."""
        self.n = state['n']
        self.summary = state['summary']

    def get_quantile_names(self):
        """Return the names of the quantiles, including 'min' and 'max'."""
        quantile_names = ['min'] + [f'p{int(round(q * 100))}' for q in self.quantiles] + ['max']
        return quantile_names


# t_digest_quantile_approximator.py
from tdigest import TDigest

class TDigestQuantileApproximator:
    """
    Implements the T-Digest algorithm for quantile estimation.
    """
    def __init__(self, quantiles):
        """
        :param quantiles: List of quantiles to estimate (e.g., [0.25, 0.5, 0.75])
        """
        self.quantiles = sorted(quantiles)
        self.reset()

    def reset(self):
        """Reset the T-Digest approximator."""
        self.digest = TDigest()

    def partial_fit(self, X):
        """Incrementally fit the model to the data."""
        for x in X:
            self.digest.update(x)
        return self

    def get_markers(self):
        """Return the quantile estimates."""
        if self.digest.n == 0:
            return None
        markers = [self.percentile(0)]  # min
        for q in self.quantiles:
            markers.append(self.percentile(q * 100))
        markers.append(self.percentile(100))  # max
        return markers

    def percentile(self, p):
        """Compute the percentile value using the digest's percentile method."""
        return self.digest.percentile(p)

    def is_initialized(self):
        """Check if the approximator has enough data."""
        return self.digest.n > 0

    def get_state(self):
        """Get the internal state for saving or exporting."""
        # Use the to_dict() method for serialization
        return self.digest.to_dict()

    def set_state(self, state):
        """Set the internal state from saved data."""
        self.digest = TDigest()
        self.digest.update_from_dict(state)

    def get_quantile_names(self):
        """Return the names of the quantiles, including 'min' and 'max'."""
        quantile_names = ['min'] + [f'p{int(round(q * 100))}' for q in self.quantiles] + ['max']
        return quantile_names

class QuantileManager:
    """
    Manages quantile approximators for different KPIs using specified algorithms and parameters.
    """
    def __init__(self, kpi_approximators):
        """
        :param kpi_approximators: Dictionary mapping KPI names to either:
            - An initialized quantile approximator instance
            - A tuple of (algorithm_class, parameters_dict)
        """
        self.quantile_approximators = {}
        for kpi, approximator_info in kpi_approximators.items():
            if isinstance(approximator_info, tuple):
                # It's a tuple of (algorithm_class, parameters_dict)
                algorithm_class, params = approximator_info
                self.quantile_approximators[kpi] = algorithm_class(**params)
            elif isinstance(approximator_info, type):
                # It's an algorithm class, instantiate with default params
                self.quantile_approximators[kpi] = algorithm_class()
            else:
                # Assume it's an initialized approximator instance
                self.quantile_approximators[kpi] = approximator_info

    def fit(self):
        for approximator in self.quantile_approximators.values():
            approximator.fit([])

    def partial_fit(self, kpi_name, value):
        """Update the quantile approximations for a specific KPI."""
        if kpi_name in self.quantile_approximators:
            self.quantile_approximators[kpi_name].partial_fit([value])

    def get_markers(self, kpi_name):
        """Get the markers (quantile estimates) for a specific KPI."""
        if kpi_name in self.quantile_approximators:
            return self.quantile_approximators[kpi_name].get_markers()
        else:
            return None

    def reset(self):
        """Reset all approximators."""
        for approximator in self.quantile_approximators.values():
            approximator.reset()

    def export_markers(self):
        """
        Export all markers and their associated metadata to a DataFrame.
        """
        export_data = []
        for kpi, approx in self.quantile_approximators.items():
            if approx.is_initialized():
                export_data.append({
                    'kpi': kpi,
                    'markers': approx.get_markers(),
                    'state': approx.get_state(),
                    'algorithm': approx.__class__.__name__
                })
        return pd.DataFrame(export_data)

    def load_markers(self, markers_df):
        """
        Load markers and their associated metadata from a DataFrame.
        """
        for _, row in markers_df.iterrows():
            kpi = row['kpi']
            if kpi in self.quantile_approximators:
                approx = self.quantile_approximators[kpi]
                approx.set_state(row['state'])

    def represent_markers(self):
        """
        Return a DataFrame with the quantile estimates for each KPI,
        with quantiles as columns named appropriately.
        """
        markers_dict = {}
        for kpi, approximator in self.quantile_approximators.items():
            markers = approximator.get_markers()
            if markers is not None:
                markers_dict[kpi] = {}
                quantile_names = approximator.get_quantile_names()
                for name, value in zip(quantile_names, markers):
                    markers_dict[kpi][name] = value
        df = pd.DataFrame.from_dict(markers_dict, orient='index')
        df.reset_index(inplace=True)
        df.rename(columns={'index': 'kpi'}, inplace=True)
        return df


In [4]:
from scipy.stats import linregress # Required for trend-based KPIs
import numpy as np
from typing import Dict, List

class Symbolizer:
    def __init__(
        self,
        quantile_manager: QuantileManager,
        kpis: Dict[str, str],
        decisions: Dict[str, str],
        kpi_change_threshold_percent: int = 10,
    ):
        """
        Initializes the Symbolizer with a QuantileManager, KPI list, and decision list.

        :param quantile_manager: An instance of QuantileManager to manage quantiles for KPIs.
        :param kpis: A dictionary mapping KPI human-readable names to environment kpi names.
        :param decisions: A dictionary mapping decision human-readable names to environment decision names.
        :param kpi_change_threshold_percent: Threshold percentage to determine if a KPI has increased or decreased.
        """
        self.quantile_manager = quantile_manager
        self.kpis = kpis
        self.decisions = decisions
        self.kpi_change_threshold_percent = kpi_change_threshold_percent

        # Define category names for standard KPIs
        self.standard_category_names = ["VeryLow", "Low", "Medium", "High", "VeryHigh"]
        # Define category names for special KPIs
        self.special_category_names = ["Low", "Medium", "High"]
        # List of KPIs that use the special three-category system
        self.special_kpis = []  # Assuming no special KPIs in Pensive

        self.array_kpis = ['dl_tput_all', 'dl_delay_all', 'bwidth_all']
        self.trend_categories = ["fluctuating", "spiking", "dropping"]

        self.slopes_hist = {'dl_tput_all': [], 'dl_delay_all': [], 'bwidth_all': []}

        # Map array KPIs to their base KPIs for quantile management
        self.array_to_base_kpi = {
            'dl_tput_all': 'dl_tput',
            'dl_delay_all': 'dl_delay',
            'bwidth_all': 'bwidth'
        }

        # Define current timestep index for each array KPI
        self.array_current_timestep_index = {
            'dl_tput_all': -1,  # Last element is current for dl_tput and dl_delay
            'dl_delay_all': -1,
            'bwidth_all': 3     # 4th element (index 3) is current for bwidth
        }


        ###########################
        # Removed hardcoded trend_thresholds
        ###########################

        self.prev_state_dict = None

    def create_symbolic_form(self, curr_step_dict, update_approximators=True):
        """
        Receives the current state of an agent at a timestep and returns the symbolic representation.
        Internally manages the previous state.

        :param curr_step_dict: Current timestep dictionary containing state information.
        :param update_approximators: Whether to update the approximators with current data.
        :return: A list of dictionaries with symbolic representations for each slice.
        """
        effects_symbolic_representation = []
        prev_step_dict = self.prev_state_dict

        if prev_step_dict is not None:
            # 1. Create symbolic rep for each KPI
            symbolic_state = self._calculate_symbolic_state(
                curr_step_dict, prev_step_dict
            )

            symbolic_representation = {
                "Timestep": curr_step_dict["Timestep"],
                **symbolic_state,
                "reward": curr_step_dict["reward"],
            }
            effects_symbolic_representation.append(symbolic_representation)
        else:
            # Handle the first timestep where there is no previous state
            symbolic_representation = {
                "Timestep": curr_step_dict["Timestep"],
                **self._initialize_symbols(curr_step_dict),
                "reward": curr_step_dict["reward"],
            }
            effects_symbolic_representation.append(symbolic_representation)

        if update_approximators:
            # Update the quantile approximators with current KPI data
            self._add_timestep_kpi_data_to_approximator(curr_step_dict)

            # Update the internal previous state with the current slice data
            self.prev_state_dict = curr_step_dict.copy()

        return effects_symbolic_representation

    def _initialize_symbols(self, curr_step_dict: dict):
      """
      Initialize symbols for the first timestep.

      :param curr_step_dict: Current step dictionary.
      :return: A dictionary with initial symbolic representations.
      """
      symbolic_representation = {}
      for key, env_name in {**self.kpis, **self.decisions}.items():
          curr_value = curr_step_dict[env_name]
          if env_name in self.kpis.values():
            symbolic_representation[env_name] = self._get_initial_symb(
                  curr_value, env_name
              )
          else:
            predicate = "const"
            symbolic_representation[env_name] = f"{predicate}({env_name}, {curr_value})"
      return symbolic_representation

    def _get_initial_symb(self, curr_value, env_name):
        """
        Generate KPI symbolic representation for the first timestep.

        :param curr_value: Current KPI value.
        :param env_name: Name of the KPI.
        :return: Symbolic representation string.
        """
        markers = self.quantile_manager.get_markers(env_name)
        if markers is None:
            return f"Unknown({env_name})"  # Handle the case where markers are not available

        category = self._get_category(curr_value, markers, env_name)
        predicate = "const"
        return f"{predicate}({env_name}, {category})"

    def _calculate_symbolic_state(self, curr_step_dict, prev_step_dict):
        """
        Calculate the symbolic state for KPIs and decisions based on current and previous values.

        :param curr_step_dict: Current dictionary.
        :param prev_step_dict: Previous dictionary.
        :return: A dictionary with symbolic representations.
        """
        symbolic_representation = {}
        for key, env_name in {**self.kpis, **self.decisions}.items():
            curr_value = curr_step_dict[env_name]
            prev_value = prev_step_dict[env_name]

            if env_name in self.kpis.values():
                symbolic_representation[env_name] = self._get_symb(
                    curr_value, prev_value, env_name
                )
            else:
                change_percentage = self._find_change_percentage(
                    curr_value, prev_value
                )
                predicate = self._get_predicate(change_percentage)

                if predicate == "const":
                    symbolic_representation[env_name] = f"{predicate}({env_name}, {curr_value})"
                else:
                    symbolic_representation[env_name] = f"{predicate}({env_name}, {prev_value}, {curr_value})"
        return symbolic_representation

    def _get_symb(self, curr_value, prev_value, kpi_name):
        """Calculate the symbolic representation of KPI changes."""
        if kpi_name in self.array_kpis:
            current_timestep_index = self.array_current_timestep_index[kpi_name]
            change_percentage = self._find_change_percentage(curr_value[current_timestep_index], prev_value[current_timestep_index])
            predicate = self._get_predicate(change_percentage)

            # Use the base KPI's markers for categorization
            base_kpi = self.array_to_base_kpi[kpi_name]
            markers = self.quantile_manager.get_markers(base_kpi)  # Use base KPI here
            if markers is None:
                return f"Unknown({kpi_name})"

            category = self._get_category(curr_value[current_timestep_index], markers, base_kpi)  # Pass base KPI here
            trend = self._analyze_trend(curr_value, kpi_name)

            return f"{predicate}({kpi_name}, {category}, {trend})"

        # Original logic for non-array KPIs
        change_percentage = self._find_change_percentage(curr_value, prev_value)
        predicate = self._get_predicate(change_percentage)
        markers = self.quantile_manager.get_markers(kpi_name)
        if markers is None:
            return f"Unknown({kpi_name})"
        category = self._get_category(curr_value, markers, kpi_name)
        return f"{predicate}({kpi_name}, {category})"

    def _get_category(self, value, markers, kpi_name):
        """Categorize a value based on percentile markers."""
        if kpi_name in self.special_kpis:
            # Use fixed percentiles for special KPIs
            if markers is None or len(markers) < 3:
                return "Unknown"

            p30 = markers[1]
            p70 = markers[2]

            if value < p30:
                return self.special_category_names[0]  # Low
            elif value <= p70:
                return self.special_category_names[1]  # Medium
            else:
                return self.special_category_names[2]  # High
        else:
            # Use standard 5 categories for other KPIs
            if markers is None or len(markers) < 5:
                return "Unknown"

            p20 = markers[1]
            p40 = markers[2]
            p60 = markers[3]
            p80 = markers[4]

            if value <= p20:
                return self.standard_category_names[0]  # VeryLow
            elif value <= p40:
                return self.standard_category_names[1]  # Low
            elif value <= p60:
                return self.standard_category_names[2]  # Medium
            elif value <= p80:
                return self.standard_category_names[3]  # High
            else:
                return self.standard_category_names[4]  # VeryHigh

    def _analyze_trend(self, value_array: List[float], kpi_name: str) -> str:
        """
        Analyzes trend in array values, ignoring zeros.
        Returns 'fluctuating', 'spiking', or 'dropping' based on dynamic quantile thresholds.

        :param value_array: List of KPI values.
        :param kpi_name: Name of the KPI being analyzed.
        :return: String representing the trend.
        """
        values = [v for v in value_array if v != 0.0]
        if len(values) < 2:
            return "fluctuating"

        x = np.arange(len(values))
        slope, _, _, _, _ = linregress(x, values)

        self.slopes_hist[kpi_name].append(slope)

        # Fetch dynamic thresholds from quantile manager for slopes
        slope_kpi_name = f"{kpi_name}_slope" # e.g., 'dl_tput_all_slope'
        slope_markers = self.quantile_manager.get_markers(slope_kpi_name)

        if slope_markers is None or len(slope_markers) < 3: # min, p20, p80, max - we expect at least min, p20, p80
            # Fallback to fixed thresholds if quantiles are not available (e.g., at the beginning)
            low_threshold = -0.0074 if kpi_name == 'dl_tput_all' else -0.0151
            high_threshold = 0.0067 if kpi_name == 'dl_tput_all' else 0.0259
        else:
            p20_threshold = slope_markers[1] # p20 - Correct index is 1
            p80_threshold = slope_markers[2] # p80 - Correct index is 2
            low_threshold = p20_threshold
            high_threshold = p80_threshold

        if slope > high_threshold:
            return "spiking"
        elif slope < low_threshold:
            return "dropping"
        else:
            return "fluctuating"

    def _find_change_percentage(self, curr_value, prev_value):
        """Calculate the change percentage of the given parameter."""
        if isinstance(curr_value, list):
            # For arrays, use the specified current timestep value
            curr = curr_value
            prev = prev_value
        else:
            curr = curr_value
            prev = prev_value

        if prev == 0:
            if curr == 0:
                return 0
            else:
                return "inf"
        else:
            return int(100 * (curr - prev) / prev)

    def _get_predicate(self, change_percentage):
        """
        Return the correct predicate according to the change percentage.

        :param change_percentage: The percentage change.
        :return: A string representing the predicate ('inc', 'dec', 'const').
        """
        if change_percentage == "inf":
            return "inc"
        elif change_percentage > self.kpi_change_threshold_percent:
            return "inc"
        elif change_percentage < -self.kpi_change_threshold_percent:
            return "dec"
        else:
            return "const"

    def _add_timestep_kpi_data_to_approximator(self, timestep_dict):
        """Adds KPI data of one timestep to the quantile approximators, including slopes for '_all' KPIs."""
        for kpi_name in self.kpis.values():
            if kpi_name not in self.array_kpis:
                self.quantile_manager.partial_fit(kpi_name, timestep_dict[kpi_name])
            else:
                current_timestep_index = self.array_current_timestep_index[kpi_name]
                self.quantile_manager.partial_fit(kpi_name, timestep_dict[kpi_name][current_timestep_index])
                # Calculate and fit slope to quantile manager
                slope_kpi_name = f"{kpi_name}_slope" # e.g., 'dl_tput_all_slope'
                if timestep_dict[kpi_name] and len(timestep_dict[kpi_name]) >= 2: # Ensure history is long enough and not empty
                    values = [v for v in timestep_dict[kpi_name] if v != 0.0] # Ignore zeros for slope calculation
                    if len(values) >= 2: # Need at least 2 non-zero values to calculate slope
                        x = np.arange(len(values))
                        slope, _, _, _, _ = linregress(x, values)
                        self.slopes_hist[kpi_name].append(slope) # Store slope values
                        self.quantile_manager.partial_fit(slope_kpi_name, slope) # Fit slope to slope's quantile manager

In [5]:

import re
from typing import Dict, Any, Optional, Tuple, List

import networkx as nx
from pyvis.network import Network

class KnowledgeGraph:
    def __init__(self, state_kpis: List[str]) -> None:
        """
        Initializes the KnowledgeGraph.
        
        Args:
            state_kpis (List[str]): A list of keys from the symbolic form dictionary that will be used to create the state and state_id.
        """
        self.G = nx.DiGraph()
        self.net = None
        self.previous_state: Optional[str] = None
        self.state_kpis = state_kpis  # Store the KPIs used for state creation
        
        if not isinstance(state_kpis, list) or not all(isinstance(kpi, str) for kpi in state_kpis):
            raise TypeError("state_kpis must be a list of strings.")
    
    def _extract_state(self, symbolic_form_dict: Dict[str, Any]) -> Tuple[Tuple, str]:
        """Extracts the current state and state_id based on the configured KPIs."""
        current_state = tuple(symbolic_form_dict.get(kpi) for kpi in self.state_kpis)
        state_id = "_".join(str(x) for x in current_state if x is not None)
        return current_state, state_id

    def _parse_action(self, action_str: str) -> Optional[int]:
        """
        Parses action string and returns:
        - 0 for const actions
        - Difference in indices between bitrates for inc/dec actions
        """
        VIDEO_BIT_RATE = [300., 750., 1200., 1850., 2850., 4300.]
        
        # Extract predicate and parameters
        predicate_match = re.match(r"(\w+)\(([^)]+)\)", action_str)
        if not predicate_match:
            return None
        
        predicate = predicate_match.group(1)
        params_str = predicate_match.group(2)
        
        # Split parameters and remove whitespace
        params = [param.strip() for param in params_str.split(',')]
        
        # Handle const action
        if predicate == "const":
            return 0
        
        # Handle inc and dec actions
        if predicate in ["inc", "dec"]:
            try:
                # Extract the bitrate values
                from_bitrate = float(params[1])
                to_bitrate = float(params[2])
                
                # Find indices in VIDEO_BIT_RATE array
                from_index = VIDEO_BIT_RATE.index(from_bitrate)
                to_index = VIDEO_BIT_RATE.index(to_bitrate)
                
                # Calculate difference in steps
                steps = to_index - from_index
                
                return steps
            except (ValueError, IndexError):
                return None
        
        return None
    
    def update_graph(self, symbolic_form: Dict[str, Any]) -> None:
        current_state, state_id = self._extract_state(symbolic_form)
        current_reward = symbolic_form.get("reward", 0)
        action = symbolic_form.get("sel_brate")

        self.G.add_node(
            state_id, 
            state=current_state, 
            occurrence=self.G.nodes.get(state_id, {}).get('occurrence', 0) + 1,
            total_reward=self.G.nodes.get(state_id, {}).get('total_reward', 0.0) + current_reward
            )
        node_data = self.G.nodes[state_id]
        node_data['mean_reward'] = node_data['total_reward'] / node_data['occurrence']

        if self.previous_state:
            edge_data = self.G.get_edge_data(self.previous_state, state_id, default={})
            edge_data['occurrence'] = edge_data.get('occurrence', 0) + 1
            edge_data['total_reward'] = edge_data.get('total_reward', 0.0) + current_reward
            edge_data['mean_reward'] = edge_data['total_reward'] / edge_data['occurrence']
            actions = edge_data.setdefault('actions', {})
            actions.setdefault(action, {'count': 0, 'total_reward': 0.0})['count'] += 1
            actions[action]['total_reward'] += current_reward
            for act_data in actions.values():
                act_data['mean_reward'] = act_data['total_reward'] / act_data['count']
                act_data['probability'] = act_data['count'] / edge_data['occurrence']
            self.G.add_edge(self.previous_state, state_id, **edge_data)

        self.previous_state = state_id
    
    def get_recommendation(self, symbolic_form: Dict[str, Any]) -> Optional[float]:
        _, state_id = self._extract_state(symbolic_form)
        # current_action = symbolic_form.get("sel_brate")
        # parsed_action = self._parse_action(current_action)
        
        # print(f"State ID: {state_id}, Current Action: {current_action}, Current Bitrate: {parsed_action}")
        
        if state_id not in self.G.nodes:
            print(f"State ID {state_id} not found in graph.")
            return None

        actions_dict = {}  # Dictionary to store all actions
        for _, next_node, edge_data in self.G.out_edges(state_id, data=True):
            if 'actions' in edge_data:
                for action, data in edge_data['actions'].items():
                    parsed_action = self._parse_action(action)
                    if parsed_action not in actions_dict:
                        actions_dict[parsed_action] = {
                            'count': data['count'],
                            'total_reward': data['total_reward']
                        }
                    # If action already exists, update count and total_reward
                    else:
                        actions_dict[parsed_action]['count'] += data['count']
                        actions_dict[parsed_action]['total_reward'] += data['total_reward']
        
        return actions_dict
        
        # return self._parse_action(best_alt_action) if best_alt_action else None

    def _update_probabilities_and_sizes(self):
        total_node_occurrence = sum(nx.get_node_attributes(self.G, 'occurrence').values())
        if total_node_occurrence == 0:
            return

        for node, data in self.G.nodes(data=True):
            prob = data['occurrence'] / total_node_occurrence
            data['probability'] = prob
            data['size'] = 30 + 120 * (prob**0.5)

        for u, v, data in self.G.edges(data=True):
            total_transitions_from_u = sum(self.G[u][nbr]['occurrence'] for nbr in self.G.successors(u))
            data['probability'] = data['occurrence'] / total_transitions_from_u if total_transitions_from_u > 0 else 0
            data['width'] = 1 + 5 * data['probability']

    def build_graph(self):
        self._update_probabilities_and_sizes()

        self.net = Network(height="1500px", width="100%", bgcolor="#222222", font_color="white",
                           directed=True, notebook=True, filter_menu=True, select_menu=True, cdn_resources="in_line")
        self.net.from_nx(self.G)

        for node in self.net.nodes:
            data = self.G.nodes[node['id']]
            node['title'] = f"State: {data.get('state')} \nOccurrence: {data.get('occurrence')} \n" \
                            f"Mean Reward: {data.get('mean_reward'):.2f} \nProbability: {100 * data.get('probability', 0):.1f}%"

        for edge in self.net.edges:
            u, v = edge['from'], edge['to']
            data = self.G[u][v]
            action_info = "\nActions:\n"
            for action, act_data in data.get('actions', {}).items():
                action_info += f"  {action}: Count: {act_data['count']}, Mean Reward: {act_data['mean_reward']:.2f}, Probability: {act_data['probability']:.2f}\n"

            edge['title'] = f"Transition: {u} -> {v}\nOccurrence: {data.get('occurrence')}\nMean Reward: {data.get('mean_reward'):.2f}\nProbability: {100 * data.get('probability', 0):.1f}%{action_info}"

        num_nodes = self.G.number_of_nodes()
        num_edges = self.G.number_of_edges()
        self.net.add_node("size_info", label=f"Nodes: {num_nodes}<br>Edges: {num_edges}", shape="text", x='-95%', y=0, physics=False)

        self.net.barnes_hut(overlap=1)
        self.net.show_buttons(filter_=['physics'])

    def get_graph(self, mode="all"):
        self.build_graph()
        return self.G, self.net

# Helper Functions

In [6]:
# -------------------- Preprocess Function (Inlined) --------------------
def preprocess(data_dict, video_bit_rate):
    """
    Preprocesses the data dictionary by extracting throughput and delay values,
    removing 'Next Chunk Sizes (Mb)', converting bitrate ratio to bitrate, and rounding
    numeric values. Also maintains historical data for throughput and delay.

    Args:
        data_dict (dict): The raw data dictionary containing state information.
        video_bit_rate (list): A list of available video bitrates.

    Returns:
        dict: The processed data dictionary with both current and historical values.
    """
    # Store historical data before processing
    dl_tput_history = data_dict['Download Chunk Throughput (Kbps/ms)'] if 'Download Chunk Throughput (Kbps/ms)' in data_dict else []
    dl_delay_history = data_dict['Download Chunk Delay (Norm by 1/10 sec)'] if 'Download Chunk Delay (Norm by 1/10 sec)' in data_dict else []

    # 1. Keep only the last value [-1] in 'Download Chunk Throughput' and 'Download Chunk Delay'
    if 'Download Chunk Throughput (Kbps/ms)' in data_dict:
        data_dict['Download Chunk Throughput (Kbps/ms)'] = data_dict['Download Chunk Throughput (Kbps/ms)'][-1]
    if 'Download Chunk Delay (Norm by 1/10 sec)' in data_dict:
        data_dict['Download Chunk Delay (Norm by 1/10 sec)'] = data_dict['Download Chunk Delay (Norm by 1/10 sec)'][-1]

    # 2. Remove 'Next Chunk Sizes (Mb)' key
    if 'Next Chunk Sizes (Mb)' in data_dict:
        del data_dict['Next Chunk Sizes (Mb)']

    # 3. Convert 'Prev Bitrate Ratio' to actual bitrate using video_bit_rate
    prev_bitrate_ratio = data_dict.get('Prev Bitrate Ratio')
    if prev_bitrate_ratio is not None:
        max_bitrate = max(video_bit_rate)
        last_brate_value = prev_bitrate_ratio * max_bitrate
    else:
        last_brate_value = None

    # 4. Rename keys to match the desired column names
    renamed_data_dict = {
        'Timestep': data_dict.get('Timestep'),
        'File_name': data_dict.get('File Name'),
        'last_brate': last_brate_value,
        'buffer': data_dict.get('Buffer Size (Norm by 1/10 sec)'),
        'dl_tput': data_dict.get('Download Chunk Throughput (Kbps/ms)'),
        'dl_delay': data_dict.get('Download Chunk Delay (Norm by 1/10 sec)'),
        'rem_chunks': data_dict.get('Chunks Remain Ratio'),
        'sel_brate': data_dict.get('Selected Bitrate (Kbps)'),
        'reward': data_dict.get('reward'),
        'bwidth': data_dict.get('Bandwidth (Kbps)')[3],
        # Add historical data
        'dl_tput_all': dl_tput_history,
        'dl_delay_all': dl_delay_history,
        'bwidth_all': data_dict.get('Bandwidth (Kbps)')
    }

    # 5. Round all numeric values to 3 decimal places for the entire dictionary
    processed_data_dict = {
        k: (
            round(v, 3) if isinstance(v, (int, float)) else
            [round(x, 3) for x in v] if isinstance(v, list) else # Handle lists
            [round(x, 3) for x in v.tolist()] if isinstance(v, np.ndarray) else # Handle NumPy arrays
            v
        ) for k, v in renamed_data_dict.items()
    }

    return processed_data_dict

# -------------------- Helper Functions (Inlined) --------------------
def bitrate_to_acceptable_converter(as_bitrate):
    return VIDEO_BIT_RATE.index(as_bitrate)

def acceptable_to_bitrate_converter(as_bitrate):
    return VIDEO_BIT_RATE.index(as_bitrate)



In [13]:
def create_visualization_folder_path(vis_name, dataset_name, specific_name=""):
    """
    Creates the folder path for storing visualization figures and ensures the path exists.
    Includes an optional specific_name in the path.

    Args:
        vis_name (str): The name of the visualization type (folder name).
        dataset_name (str): The name of the dataset (subfolder name).
        specific_name (str, optional): An optional specific name to include in the folder path. Defaults to "".

    Returns:
        str: The full path to the visualization folder.
    """
    base_dir = "visualizations"
    folder_path = os.path.join(base_dir, vis_name, dataset_name)
    if specific_name:  # Add specific_name to path if it's not empty
        folder_path = os.path.join(folder_path, specific_name)
    os.makedirs(folder_path, exist_ok=True)  # Create directory if it doesn't exist
    return folder_path

def setup_environment(dataset='car', test_traces_folder=None):
    """
    Sets up the ABR environment for testing or running inference with forecaster.

    Args:
        dataset (str): Name of the dataset to use. Can be one of the datasets, 'train', or 'all_test'.
        test_traces_folder (str): Optional path to custom test traces.

    Returns:
        ABREnv: The initialized ABR environment.
    """
    if test_traces_folder is None:
        if dataset == 'train':
            # Load from the train_all_files folder
            test_traces_folder = ADG_SKYNET_ROOT_ADDR + '/pensive/train_all_files/'
        elif dataset == 'all_test':
            # Load from the test_all_files folder
            test_traces_folder = ADG_SKYNET_ROOT_ADDR + '/pensive/test_all_files/'
        else:
            # Load from the test_grouped_files folder
            test_traces_folder = ADG_SKYNET_ROOT_ADDR + f'/pensive/test_grouped_files/{dataset}/'

    # Always run in inference mode (train=False) with forecaster setup
    env = ABREnv(train=False, TEST_TRACES=test_traces_folder, FORECASTER_MODEL=FORECASTER_MODEL, FORECASTER_PATH=FORECASTER_PATH, if_MLP=if_MLP)
    return env

def setup_agent(checkpoint_epoch):
    """
    Sets up the PPO agent and loads a pre-trained model based on epoch checkpoint.

    Args:
        checkpoint_epoch (str): The epoch number of the checkpoint to load.

    Returns:
        network.Network: The initialized PPO agent, or None if model not found.
    """
    actor = network.Network(state_dim=S_DIM, action_dim=A_DIM, learning_rate=ACTOR_LR_RATE)
    models_dir = os.path.join(ADG_SKYNET_ROOT_ADDR, 'pensive', 'pensive-gamma', 'models') # Corrected models path

    model_filename = None
    for filename in os.listdir(models_dir):
        if f"confc4_model_t3_ep_{checkpoint_epoch}" in filename:
            model_filename = filename
            break

    if model_filename:
        model_path = os.path.join(models_dir, model_filename)
        print(f"Attempting to load agent model from: {model_path}")
        try:
            actor.load_model(model_path, weights_only=True)  # Pass weights_only=True here
            print('PPO agent model loaded successfully with weights_only=True.')
            return actor
        except Exception as e:
            print(f"Error loading model with weights_only=True: {e}")
            print("Attempting to load without weights_only (not recommended for security)...")
            try:
                actor.load_model(model_path)
                print("Model loaded without weights_only. Ensure the model source is trusted.")
                return actor
            except Exception as e:
                print(f"Error loading model: {e}")
                return None
    else:
        print(f"Error: No model checkpoint found for epoch: {checkpoint_epoch} in {models_dir}")
        return None

def setup_symbolic_components(quantile_markers_path="./norway_markers_export.csv", kpi_change_threshold=10):
    """
    Sets up the symbolic components: QuantileManager, Symbolizer, and KnowledgeGraph.

    Args:
        quantile_markers_path (str): Path to the CSV file with quantile markers.
        kpi_change_threshold (int): Threshold for KPI change percentage.

    Returns:
        tuple: QuantileManager, Symbolizer, and KnowledgeGraph objects.
    """
    kpis = {
        'last_brate': 'last_brate',
        'buffer': 'buffer',
        'dl_tput': 'dl_tput',
        'dl_delay': 'dl_delay',
        'rem_chunks': 'rem_chunks',
        'bwidth': 'bwidth',
        # Add new array KPIs
        'dl_tput_all': 'dl_tput_all',
        'dl_delay_all': 'dl_delay_all',
        'bwidth_all': 'bwidth_all',
    }

    decisions = {
        'sel_brate': 'sel_brate'
    }

    # Define the KPI approximators
    kpi_approximators = {
        'last_brate': (
            TDigestQuantileApproximator,
            {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        ),
        'buffer': (
            TDigestQuantileApproximator,
            {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        ),
        'dl_tput': (
            TDigestQuantileApproximator,
            {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        ),
        'dl_delay': (
            TDigestQuantileApproximator,
            {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        ),
        'rem_chunks': (
            TDigestQuantileApproximator,
            {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        ),
        'bwidth': (
            TDigestQuantileApproximator,
            {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        ),
        # 'dl_tput_all': ( # No QuantileApproximator for dl_tput_all and dl_delay_all and bwidth_all
        #     TDigestQuantileApproximator,
        #     {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        # ),
        # 'dl_delay_all': (
        #     TDigestQuantileApproximator,
        #     {'quantiles': [0.2, 0.4, 0.6, 0.8]}
        # ),
        # New: Quantile Managers for Slopes
        'dl_tput_all_slope': (  # QM for dl_tput_all slopes
            TDigestQuantileApproximator,
            {'quantiles': [0.4, 0.6]} # Using p20 and p80 for thresholds
        ),
        'dl_delay_all_slope': ( # QM for dl_delay_all slopes
            TDigestQuantileApproximator,
            {'quantiles': [0.4, 0.6]} # Using p20 and p80 for thresholds
        ),
        'bwidth_all_slope': ( # QM for bwidth_all slopes
            TDigestQuantileApproximator,
            {'quantiles': [0.4, 0.6]} # Using p20 and p80 for thresholds
        ),
    }

    qm = QuantileManager(kpi_approximators)
    qm.reset()

    symbolizer = Symbolizer(quantile_manager=qm, kpis=kpis, decisions=decisions, kpi_change_threshold_percent=kpi_change_threshold)

    # Define which KPIs to use for the Knowledge Graph state
    rt_kg = { # Define rt_kg dictionary for each KPI
        'last_brate': KnowledgeGraph(["last_brate"]),
        "buffer": KnowledgeGraph(["buffer"]),
        "dl_tput": KnowledgeGraph(["dl_tput"]),
        "dl_delay": KnowledgeGraph(["dl_delay"]),
        "rem_chunks": KnowledgeGraph(["rem_chunks"]),
        "bwidth": KnowledgeGraph(["bwidth"]),
        "dl_tput_all": KnowledgeGraph(["dl_tput_all"]),
        "dl_delay_all": KnowledgeGraph(["dl_delay_all"]),
        "bwidth_all": KnowledgeGraph(["bwidth_all"]),
    }

    print('Symbolic components initialized.')
    return qm, symbolizer, rt_kg

def run_agent(env, actor, symbolizer, rt_kg):
    """
    Runs the agent in the environment, collects data, and updates Knowledge Graph.

    Args:
        env (ABREnv): The ABR environment.
        actor (network.Network): The PPO agent.
        symbolizer (Symbolizer): The Symbolizer object.
        rt_kg (KnowledgeGraph): The KnowledgeGraph dictionary.

    Returns:
        tuple: DataFrames for raw and symbolic data, arrays for scores, average bitrates, total rebuffer, and markers_df.
    """
    exec_times = []
    symbolic_df = []
    raw_df = []
    marker_df_list = [] # Initialize list for marker dataframes
    avg_bitrates = np.zeros([len(env.all_file_names)])
    total_rebuff = np.zeros([len(env.all_file_names)])
    score = np.zeros([len(env.all_file_names)])
    video_count = 0
    internal_count = 0
    done = False

    obs, info = env.reset()

    while True:
        obs_tensor = torch.from_numpy(obs).float().to('cuda')
        start_time = time.time()
        action_prob_np = actor.predict(obs_tensor.unsqueeze(0)) # Get NumPy array
        end_time = time.time()
        exec_times.append(end_time - start_time)
        
        action_prob = torch.tensor(action_prob_np) # Convert NumPy array to PyTorch Tensor
        bit_rate = torch.argmax(torch.log(action_prob)).item()
        
        row_data = {
            "Timestep": internal_count,
            'File Name': env.all_file_names[video_count],  # File name for the video
            'Prev Bitrate Ratio': obs[0, -1],  # s_batch[:, 0, -1]
            'Buffer Size (Norm by 1/10 sec)': obs[1, -1],  # s_batch[:, 1, -1]
            'Download Chunk Throughput (Kbps/ms)': obs[2, :].tolist(),  # s_batch[:, 2, :]
            'Download Chunk Delay (Norm by 1/10 sec)': obs[3, :].tolist(),  # s_batch[:, 3, :]
            'Bandwidth (Kbps)': obs[4, :],  # s_batch[:, 4, :]
            'Next Chunk Sizes (Mb)': obs[5, :6].tolist(),  # s_batch[:, 4, :6]
            'Chunks Remain Ratio': obs[6, -1],  # s_batch[:, 5, -1]
        }

        obs, rew, done, _, info = env.step(bit_rate)

        row_data.update({
            'Selected Bitrate (Kbps)': info['bitrate'],
            'reward': rew,
        })

        processed_data = preprocess(row_data, VIDEO_BIT_RATE)
        symbolic_form = symbolizer.create_symbolic_form(processed_data)

        if not any("unknown" in str(v).lower() for v in symbolic_form[0].values()):

            raw_df.append(processed_data)
            # Add 'File Name' to symbolic data
            for sym_row in symbolic_form:
                sym_row['File_name'] = env.all_file_names[video_count]

            symbolic_df.extend(symbolic_form)

            for key in rt_kg: # Update each KG in the dictionary
                if key != "combined": # combined KG is updated with combined state
                    rt_kg[key].update_graph(symbolic_form[0])
                else:
                    rt_kg[key].update_graph(symbolic_form[0]) # combined KG update

            # Get the markers from the quantile manager and append to the list
            marker_data = symbolizer.quantile_manager.represent_markers()
            marker_data['Timestep'] = internal_count
            marker_data['File_name'] = env.all_file_names[video_count] #Add file name here
            marker_df_list.append(marker_data)

        score[video_count] += rew
        avg_bitrates[video_count] += info['bitrate']
        total_rebuff[video_count] += info['rebuffer']

        print(f"timestamp: {internal_count} | File: {env.all_file_names[video_count]} | Chunks: {obs[5,-1]} | rew: {rew:.3f} | score: {score[video_count]:.3f}")
        print(f"Selected Bitrate: {VIDEO_BIT_RATE[bit_rate]} | Rebuffer: {info['rebuffer']:.3f}")
        print(info)
        print("-" * 100)

        internal_count += 1

        if done:
            avg_bitrates[video_count] /= 48.0
            video_count += 1
            if video_count >= len(env.all_file_names):
                break
            obs, info = env.reset()

    return pd.DataFrame(raw_df), pd.DataFrame(symbolic_df), score, avg_bitrates, total_rebuff, pd.concat(marker_df_list), exec_times

# Run the agent's code

In [14]:
# --- Configuration ---
available_datasets = ['bad', 'bus', 'car', 'ferry', 'metro', 'synthetic', 'tram', 'train', 'all_test'] # List of datasets to choose from
dataset = 'car' # Choose a dataset from the list, default 'car'

if dataset not in available_datasets:
    raise ValueError(f"Selected dataset '{dataset}' is not in the available datasets list: {available_datasets}")

checkpoint_epochs = ['300', '350', '400', '450', '500', '550', '550', '600', '650', '700', '750', '800', '850', '900', '950', '1000', '1050', '1100', '1150', '1200', '1250', '1300', '1350', '1400', '1450', '1500']
selected_checkpoint_epoch = '550' # Choose a checkpoint epoch from the list

if selected_checkpoint_epoch not in checkpoint_epochs:
    raise ValueError(f"Selected checkpoint epoch '{selected_checkpoint_epoch}' is not in the available checkpoint epochs list: {checkpoint_epochs}")


# --- Setup ---
print("-" * 30)
env = setup_environment(dataset) # Call setup_environment with dataset
print(f"Loading dataset: {dataset}")

actor = setup_agent(selected_checkpoint_epoch) # Call setup_agent with checkpoint epoch
if actor is None:
    sys.exit("Agent setup failed, exiting.") # Exit if agent setup fails

print("-" * 30)

# --- Run Agent ---
if dataset == 'train':
    qm, symbolizer, rt_kg = setup_symbolic_components() # Call setup_symbolic_components
    print("Symbolic components setup completed.")
    train_raw_df, train_symbolic_df, score, avg_bitrates, total_rebuff, markers_df, exec_times = run_agent(env, actor, symbolizer, rt_kg) # Call run_agent
    print("Agent run completed for TRAIN dataset.")
else:
    qm, symbolizer, rt_kg = setup_symbolic_components() # Call setup_symbolic_components
    print("Symbolic components setup completed.")
    raw_df, symbolic_df, score, avg_bitrates, total_rebuff, markers_df, exec_times = run_agent(env, actor, symbolizer, rt_kg) # Call run_agent
    print("Agent run completed for TEST dataset.")


# # --- Display Results ---
# print(f"Total score: {np.sum(score):.3f}")
# print(f"Average bitrate: {np.mean(avg_bitrates):.3f}")
# print(f"Total rebuffer: {np.sum(total_rebuff):.3f}")

------------------------------
Loading forecaster model from: /home/erfan/data/mobicom-2024/pensive/pensive-gamma/TST-models/tst3_epoch_1200_loss_0.1210.pt
Loading dataset: car
Attempting to load agent model from: /home/erfan/data/mobicom-2024/pensive/pensive-gamma/models/confc4_model_t3_ep_550000_6378.338.pth
PPO agent model loaded successfully with weights_only=True.
------------------------------
Symbolic components initialized.
Symbolic components setup completed.
timestamp: 0 | File: norway_car_1 | Chunks: 0.15558000000000002 | rew: -0.150 | score: -0.150
Selected Bitrate: 300.0 | Rebuffer: 0.000
{'bitrate': 300.0, 'rebuffer': 0.0}
----------------------------------------------------------------------------------------------------
timestamp: 1 | File: norway_car_1 | Chunks: 0.139857 | rew: 0.300 | score: 0.150
Selected Bitrate: 300.0 | Rebuffer: 0.000
{'bitrate': 300.0, 'rebuffer': 0.0}
-----------------------------------------------------------------------------------------------

In [79]:
def aggregate_recommendations(rt_kg, symbolic_form):
    aggregated_actions = {}
    
    for key in rt_kg:
        # print(f"get recommendation for {key} KG: {symbolic_form[key]}")
        action_dict = rt_kg[key].get_recommendation(symbolic_form)
        # print(f"Action dict for {key} KG: {action_dict}")
        
        # Skip if no recommendations were found
        if action_dict is None:
            continue
        
        # Aggregate the actions
        for action, data in action_dict.items():
            if action not in aggregated_actions:
                # Add new action to aggregated dictionary
                aggregated_actions[action] = {
                    'count': data['count'],
                    'total_reward': data['total_reward']
                }
            else:
                # Update existing action in aggregated dictionary
                aggregated_actions[action]['count'] += data['count']
                aggregated_actions[action]['total_reward'] += data['total_reward']
    
    # Calculate mean reward for each aggregated action
    for action in aggregated_actions:
        count = aggregated_actions[action]['count']
        total = aggregated_actions[action]['total_reward']
        aggregated_actions[action]['mean_reward'] = total / count if count > 0 else 0
    
    return aggregated_actions

import math

def select_best_action(aggregated_recommendations, mode='max_reward', confidence_factor=1.0):
    """
    Select the best action based on specified mode.
    
    Parameters:
    - aggregated_recommendations: Dictionary of actions with statistics
    - mode: 'max_reward' or 'highest_confidence'
    - confidence_factor: Factor to adjust confidence calculation (higher = more exploration)
    
    Returns:
    - Best action based on selected mode
    """
    if not aggregated_recommendations:
        return None
    
    best_action = None
    best_value = float('-inf')
    
    for action, data in aggregated_recommendations.items():
        # Extract values
        count = data['count']
        mean_reward = data['mean_reward']
        
        if mode == 'max_reward':
            # Simply use mean reward
            value = mean_reward
        elif mode == 'highest_confidence':
            # Calculate confidence score
            # Using Upper Confidence Bound (UCB) formula
            if count > 0:
                # Higher count means more confidence in the estimate
                confidence_score = mean_reward + (confidence_factor * math.sqrt(math.log(sum(d['count'] for d in aggregated_recommendations.values())) / count))
            else:
                confidence_score = float('-inf')
            value = confidence_score
        else:
            raise ValueError(f"Unknown mode: {mode}")
        
        # Update best action if current is better
        if value > best_value:
            best_value = value
            best_action = action
    
    return best_action



In [80]:
symbolic_df = []
raw_df = []
marker_df_list = [] # Initialize list for marker dataframes
avg_bitrates = np.zeros([len(env.all_file_names)])
total_rebuff = np.zeros([len(env.all_file_names)])
score = np.zeros([len(env.all_file_names)])
video_count = 0
internal_count = 0
done = False

obs, info = env.reset()

while True:
    obs_tensor = torch.from_numpy(obs).float().to('cuda')
    action_prob_np = actor.predict(obs_tensor.unsqueeze(0)) # Get NumPy array
    action_prob = torch.tensor(action_prob_np) # Convert NumPy array to PyTorch Tensor
    bit_rate = torch.argmax(torch.log(action_prob)).item()


    row_data = {
        "Timestep": internal_count,
        'File Name': env.all_file_names[video_count],  # File name for the video
        'Prev Bitrate Ratio': obs[0, -1],  # s_batch[:, 0, -1]
        'Buffer Size (Norm by 1/10 sec)': obs[1, -1],  # s_batch[:, 1, -1]
        'Download Chunk Throughput (Kbps/ms)': obs[2, :].tolist(),  # s_batch[:, 2, :]
        'Download Chunk Delay (Norm by 1/10 sec)': obs[3, :].tolist(),  # s_batch[:, 3, :]
        'Bandwidth (Kbps)': obs[4, :],  # s_batch[:, 4, :]
        'Next Chunk Sizes (Mb)': obs[5, :6].tolist(),  # s_batch[:, 4, :6]
        'Chunks Remain Ratio': obs[6, -1],  # s_batch[:, 5, -1]
    }

    obs, rew, done, _, info = env.step(bit_rate)

    row_data.update({
        'Selected Bitrate (Kbps)': info['bitrate'],
        'reward': rew,
    })

    processed_data = preprocess(row_data, VIDEO_BIT_RATE)
    symbolic_form = symbolizer.create_symbolic_form(processed_data)
    if not any("unknown" in str(v).lower() for v in symbolic_form[0].values()):

        raw_df.append(processed_data)
        # Add 'File Name' to symbolic data
        for sym_row in symbolic_form:
            sym_row['File_name'] = env.all_file_names[video_count]

        symbolic_df.extend(symbolic_form)
        
        aggregated_dict = aggregate_recommendations(rt_kg, symbolic_form[0])
        print(f"Aggregated recommendations: {aggregated_dict}")
        best_action = select_best_action(aggregated_dict, mode='max_reward', confidence_factor=1.0)
        print(f"agent action: {symbolic_form[0]['sel_brate']}, best action: {best_action}")
        
        for key in rt_kg: # Update each KG in the dictionary
            if key != "combined": # combined KG is updated with combined state
                rt_kg[key].update_graph(symbolic_form[0])
                

    score[video_count] += rew
    avg_bitrates[video_count] += info['bitrate']
    total_rebuff[video_count] += info['rebuffer']

    print(f"timestamp: {internal_count} | File: {env.all_file_names[video_count]} | Chunks: {obs[5,-1]} | rew: {rew:.3f} | score: {score[video_count]:.3f}")
    print(f"Selected Bitrate: {VIDEO_BIT_RATE[bit_rate]} | Rebuffer: {info['rebuffer']:.3f}")
    print(info)
    print("-" * 100)

    internal_count += 1

    if done:
        avg_bitrates[video_count] /= 48.0
        video_count += 1
        if video_count >= len(env.all_file_names):
            break
        obs, info = env.reset()

State ID: inc(last_brate, Low), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: dec(buffer, VeryLow), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: inc(dl_tput, VeryHigh), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: dec(dl_delay, VeryLow), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: inc(rem_chunks, VeryHigh), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: inc(bwidth, VeryHigh), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: inc(dl_tput_all, VeryHigh, fluctuating), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: dec(dl_delay_all, VeryLow, fluctuating), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
State ID: inc(bwidth_all, VeryHigh, spiking), Current Action: const(sel_brate, 300.0), Current Bitrate: 0
Aggregated recommendations: {0: {'count': 2248, 'total_reward': 2540.2859999999996, 'mean_reward': 1.130020462

# Visualize the agent's performance - Old

## Numerical KPI per video per KPI

In [ ]:
def plot_kpi_metrics_separate_figures(raw_df, kpi_columns, titles):
    """
    Plots each KPI metric in a separate figure over time for each file in raw_df.

    Parameters:
    raw_df (pd.DataFrame): The dataframe containing the raw data with KPI metrics.
    kpi_columns (list): List of column names in raw_df to be plotted as KPIs.
    titles (list): List of titles corresponding to each KPI for separate figures.
    """
    # Validate input lengths
    if len(kpi_columns) != len(titles):
        raise ValueError("The length of kpi_columns and titles must be the same.")

    # Get unique file names
    unique_files = raw_df['File_name'].unique()

    # Define a color palette
    color_palette = qualitative.Plotly

    # Create a color map for each unique file
    file_colors = {file: color_palette[i % len(color_palette)] for i, file in enumerate(unique_files)}

    # Iterate through each KPI and create a separate figure
    for i, (kpi, title) in enumerate(zip(kpi_columns, titles)):
        fig = go.Figure()

        # Add traces for each file for the current KPI
        for file in unique_files:
            df_file = raw_df[raw_df['File_name'] == file].copy()
            df_file['Relative_Timestep'] = df_file['Timestep'] - df_file['Timestep'].min()
            fig.add_trace(
                go.Scatter(
                    x=df_file['Relative_Timestep'],
                    y=df_file[kpi],
                    mode='lines',
                    name=file,
                    legendgroup=file,
                    line=dict(color=file_colors[file])
                )
            )

        # Update layout for better visuals with vertical lines on hover and full width
        fig.update_layout(
            height=600, # Adjusted height for single plots
            autosize=True,
            title_text=title + " Over Time", # Dynamic title
            hovermode='x unified',
            yaxis_title=title, # Y-axis title is the KPI title
            xaxis_title="Relative Timestep", # X-axis title
            margin=dict(l=50, r=50, t=100, b=50)
        )

        # Add spikes to x-axis for hover effects
        fig.update_xaxes(showspikes=True, spikecolor="grey", spikethickness=1)

        # Display the interactive plot for the current KPI
        fig.show()

# Example usage:
# Assuming raw_df is already loaded as a pandas DataFrame
kpi_columns = ['buffer', 'dl_tput', 'dl_delay', 'rem_chunks', 'sel_brate', 'reward', 'bwidth'] # Added 'bwidth'
titles = ['Buffer Size', 'Download Throughput', 'Download Delay', 'Remaining Chunks', 'Selected Bitrate', 'Reward', 'Bandwidth'] # Added 'Bandwidth'
plot_kpi_metrics_separate_figures(raw_df, kpi_columns, titles)

## Markers and the approximator effect

In [ ]:
def plot_kpis_with_markers(raw_df, markers_df, timestep_col, file_name_col, kpis, marker_kpi_col='kpi', title=None):
    """
    Plots KPIs with markers in an n*1 subplot format.

    Parameters:
        raw_df (pd.DataFrame): The dataframe containing the main KPI data.
        markers_df (pd.DataFrame): The dataframe containing the marker data (e.g., percentiles).
        timestep_col (str): The name of the column representing the timestep in both dataframes.
        file_name_col (str): The name of the column representing the file name in both dataframes.
        kpis (list): A list of KPI column names to plot from raw_df.
        marker_kpi_col (str): The name of the column in markers_df that identifies the KPI (default: 'kpi').
        title (str): The title of the plot. If None, a default title will be used.
    """
    n_kpis = len(kpis)

    # Create a subplot with n_kpis rows and 1 column
    fig = make_subplots(rows=n_kpis, cols=1, subplot_titles=kpis, shared_xaxes=True, vertical_spacing=0.05)

    # Get unique file names and sort them naturally
    file_names = natsorted(raw_df[file_name_col].unique())  # Use natsorted for natural sorting

    # Assign colors using the qualitative color palette
    colors = qualitative.Plotly

    # Create a color mapping for each file_name
    color_mapping = {file_name: colors[i % len(colors)] for i, file_name in enumerate(file_names)}

    # Function to lighten a color (simple approach, can be adjusted)
    def lighten_color(color, factor=0.5):
        return f"rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}, {factor})"

    # Plot each KPI
    for i, kpi in enumerate(kpis):
        for j, file_name in enumerate(file_names):
            # Filter raw_df for the current file_name
            df_filtered = raw_df[raw_df[file_name_col] == file_name]
            color = color_mapping[file_name]  # Use the same color for the same file_name across subplots
            light_color = lighten_color(color, factor=0.5)  # Lighten the color for markers

            # Plot the main KPI trace
            fig.add_trace(
                go.Scatter(
                    x=df_filtered[timestep_col],
                    y=df_filtered[kpi],
                    mode='lines',
                    name=file_name,
                    line=dict(color=color),
                    legendgroup=file_name,  # Group traces by file_name
                    showlegend=(i == 0)  # Show legend only for the first subplot
                ),
                row=i+1, col=1
            )

            # Filter markers_df for the current file_name and KPI
            markers_filtered = markers_df[(markers_df[file_name_col] == file_name) & (markers_df[marker_kpi_col] == kpi)]
            
            # Plot the marker traces (min, p20, p40, p60, p80, max)
            for marker in ['min', 'p20', 'p40', 'p60', 'p80', 'max']:
                fig.add_trace(
                    go.Scatter(
                        x=markers_filtered[timestep_col],
                        y=markers_filtered[marker],
                        mode='lines',
                        line=dict(color=light_color, dash='dash'),
                        name=f'{file_name} {marker}',
                        legendgroup=file_name,  # Group markers with their main trace
                        showlegend=False  # Hide legend for markers
                    ),
                    row=i+1, col=1
                )

    # Update layout
    fig.update_layout(
        height=300 * n_kpis,
        title_text=title if title else "KPIs Over Time by File Name with Markers",
        title_font=dict(size=20, family="Arial", color="black", weight="bold"),  # Make title bold and centered
        title_x=0.5,  # Center the title
        legend_title="File Name",
        hovermode='x unified'  # Enable vertical hovering across all subplots
    )

    # Show the plot
    fig.show()


# Example usages
plot_kpis_with_markers(
    raw_df=raw_df,
    markers_df=markers_df,
    timestep_col='Timestep',
    file_name_col='File_name',
    kpis=['last_brate', 'buffer', 'dl_tput', 'dl_delay', 'rem_chunks', 'bwidth'],
    marker_kpi_col='kpi',  # Optional, default is 'kpi'
    title="<b>KPI's actual values and corresponding marker's values over time (Psquare algo as approximator)</b>"  # Add a custom title (optional)
)

### Temporal

In [ ]:
def create_kpi_figure_multi_file(symbolic_df, kpi_to_plot, num_files_to_plot=None):
    """
    Creates a figure for a single KPI, showing how the symbolic representations
    evolve over time, grouped by file name.

    Args:
        symbolic_df (pd.DataFrame): The input DataFrame with symbolic data.
        kpi_to_plot (str): The KPI name to plot.
        num_files_to_plot (int, optional): Number of unique files to plot. If None, plots all files.

    Returns:
        go.Figure: The created figure object
    """
    # Get unique values and sort them according to our criteria
    unique_values = symbolic_df[kpi_to_plot].unique()
    sorted_values = sorted(
        unique_values,
        key=lambda x: (
            {'VeryHigh': 5, 'High': 4, 'Medium': 3, 'Low': 2, 'VeryLow': 1}.get(re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x).group(), 0),
            {'inc': 3, 'const': 2, 'dec': 1}.get(re.search(r'(inc|const|dec)', x).group() if re.search(r'(inc|const|dec)', x) else 'const', 2)
        )
    )

    # Create value to position mapping
    current_pos = 0
    value_to_position = {}
    last_category = None

    for value in sorted_values:
        category = re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', value).group() if re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', value) else None

        if last_category is None or category != last_category:
            if last_category is not None:
                current_pos += 2
            last_category = category

        value_to_position[value] = current_pos
        current_pos += 1

    # Create the plot
    fig = go.Figure()
    colors = qualitative.Plotly  # Use Plotly's qualitative color palette

    # Get unique file names and sort them naturally
    unique_files = natsorted(symbolic_df['File_name'].unique())

    # Limit the number of files to plot if num_files_to_plot is specified
    if num_files_to_plot is not None:
        unique_files = unique_files[:num_files_to_plot]

    # Add traces for each file
    for file_idx, file in enumerate(unique_files):
        file_df = symbolic_df[symbolic_df['File_name'] == file].copy()
        # Calculate relative timestep for each file
        file_df['Relative_Timestep'] = file_df['Timestep'] - file_df['Timestep'].min()
        file_df['y_position'] = file_df[kpi_to_plot].map(value_to_position)

        fig.add_trace(go.Scatter(
            x=file_df['Relative_Timestep'],
            y=file_df['y_position'],
            mode='lines',
            name=file,
            line=dict(color=colors[file_idx % len(colors)], width=2)  # Thicker lines for better visibility
        ))

    # Update layout
    fig.update_layout(
        title=dict(
            text=f'{kpi_to_plot} over Time',
            font=dict(size=24, family="Arial", color="black", weight="bold"),
            x=0.5  # Center the title
        ),
        xaxis_title=dict(
            text='Timestep',
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        yaxis_title=dict(
            text="Symbolic Value",
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        hovermode='x unified',
        height=1000,  # Increased height significantly
        margin=dict(l=250, r=200, t=100, b=100),  # Increased left margin significantly
        font=dict(size=16, family="Arial"),
        legend=dict(
            font=dict(size=14, family="Arial"),
            yanchor="top",
            y=1,  # Position legend at the top-right
            xanchor="left",
            x=1.05,  # Move legend to the right of the plot
            orientation="v"  # Vertical orientation for better readability
        ),
        yaxis=dict(
            tickmode='array',
            ticktext=sorted_values,
            tickvals=[value_to_position[v] for v in sorted_values],
            tickfont=dict(size=12, family="Arial"), # Slightly reduced font size (optional)
            # tickangle=-45, # Removed rotation
            showgrid=True,  # Add gridlines for better readability
            gridcolor='lightgray',
            automargin=True, # Let Plotly auto-adjust margins
        ),
        xaxis=dict(
            tickfont=dict(size=16, family="Arial"),
            showgrid=True,  # Add gridlines for better readability
            gridcolor='lightgray'
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',  # Light gray background for the plot region
        paper_bgcolor='white'  # White background for the entire plot
    )

    return fig

# Plot each KPI in a separate figure
# kpi_columns_to_plot = ['buffer', 'dl_tput', 'dl_delay', 'rem_chunks', 'bwidth', 'bwidth_all'] # Added 'bwidth_all' to the list
kpi_columns_to_plot = ['bwidth', 'bwidth_all'] # Added 'bwidth_all' to the list
num_files_to_plot = 12  # Set this to the number of files you want to plot (e.g., 5, 2, or None for all)
for kpi in kpi_columns_to_plot:
    fig = create_kpi_figure_multi_file(symbolic_df, kpi, num_files_to_plot)
    fig.show()

### Probabilistic

In [ ]:
def create_probability_distribution_figure(symbolic_df, kpi_to_plot):
    """
    Creates a figure showing the probability distribution of symbolic values for a given KPI,
    grouped by file name, using a line plot.
    
    Args:
        symbolic_df (pd.DataFrame): The input DataFrame with symbolic data.
        kpi_to_plot (str): The KPI name to plot.
        
    Returns:
        go.Figure: The created figure object
    """
    # Get unique values and sort them according to our criteria
    unique_values = symbolic_df[kpi_to_plot].unique()
    sorted_values = sorted(
        unique_values,
        key=lambda x: (
            {'VeryHigh': 5, 'High': 4, 'Medium': 3, 'Low': 2, 'VeryLow': 1}.get(re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x).group(), 0),
            {'inc': 3, 'const': 2, 'dec': 1}.get(re.search(r'(inc|const|dec)', x).group() if re.search(r'(inc|const|dec)', x) else 'const', 2)
        )
    )

    # Create value to position mapping
    value_to_position = {value: idx for idx, value in enumerate(sorted_values)}

    # Create the plot
    fig = go.Figure()
    colors = qualitative.Plotly  # Use Plotly's qualitative color palette

    # Get unique file names and sort them naturally
    unique_files = natsorted(symbolic_df['File_name'].unique())

    # Add traces for each file
    for file_idx, file in enumerate(unique_files):
        file_df = symbolic_df[symbolic_df['File_name'] == file].copy()
        # Calculate the probability distribution
        value_counts = file_df[kpi_to_plot].value_counts(normalize=True)
        probabilities = [value_counts.get(value, 0) for value in sorted_values]

        fig.add_trace(go.Scatter(
            x=[value_to_position[v] for v in sorted_values],
            y=probabilities,
            mode='lines+markers',
            name=file,
            line=dict(color=colors[file_idx % len(colors)], width=2),
            marker=dict(size=8)
        ))

    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Probability Distribution of {kpi_to_plot}',
            font=dict(size=24, family="Arial", color="black", weight="bold"),
            x=0.5  # Center the title
        ),
        xaxis_title=dict(
            text='Symbolic Value',
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        yaxis_title=dict(
            text="Probability",
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        hovermode='x unified',
        height=600,  # Adjusted height for better readability
        margin=dict(l=100, r=200, t=100, b=100),  # Increased right margin for legend
        font=dict(size=16, family="Arial"),
        legend=dict(
            font=dict(size=14, family="Arial"),
            yanchor="top",
            y=1,  # Position legend at the top-right
            xanchor="left",
            x=1.05,  # Move legend to the right of the plot
            orientation="v"  # Vertical orientation for better readability
        ),
        yaxis=dict(
            tickfont=dict(size=14, family="Arial"),
            tickangle=0,  # No rotation needed
            showgrid=True,  # Add gridlines for better readability
            gridcolor='lightgray',
            range=[0, 1]  # Set y-axis range from 0 to 1
        ),
        xaxis=dict(
            tickmode='array',
            ticktext=sorted_values,
            tickvals=[value_to_position[v] for v in sorted_values],
            tickfont=dict(size=14, family="Arial"),
            tickangle=45,  # Rotate x-axis labels for better readability
            showgrid=True,  # Add gridlines for better readability
            gridcolor='lightgray'
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',  # Light gray background for the plot region
        paper_bgcolor='white'  # White background for the entire plot
    )
    
    return fig

# Plot each KPI in a separate figure
kpi_columns_to_plot = ['bwidth', 'bwidth_all']
for kpi in kpi_columns_to_plot:
    fig = create_probability_distribution_figure(symbolic_df, kpi)
    fig.show()

In [ ]:
#### Helper Functions for Statistical Measures ####
def calculate_cramers_v(confusion_matrix):
    """
    Calculates Cramer's V for a given confusion matrix.
    """
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.values.sum()
    min_dim = min(confusion_matrix.shape) - 1
    if min_dim == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))

def extract_bitrate_value(sel_brate_str):
    """
    Extracts the numerical bitrate value from the symbolic 'sel_brate' string,
    getting the LAST numerical value if multiple are present.
    """
    matches = re.findall(r'[-+]?\d*\.\d+|\d+', sel_brate_str) # Find all numerical values
    if matches:
        return float(matches[-1])  # Return the last matched number as float
    else:
        return np.nan  # Return NaN if no number is found (for robustness)

def calculate_correlation_ratio(df, row_kpi_col, col_kpi_col):
    """
    Calculates the Correlation Ratio (Eta) for categorical row_kpi_col and numerical col_kpi_col.
    """
    # Ensure col_kpi_col is numeric by extracting bitrate values
    df['numerical_sel_brate'] = df[col_kpi_col].apply(extract_bitrate_value)
    overall_variance = df['numerical_sel_brate'].var()
    if overall_variance == 0:
        return 0.0

    between_variance = 0.0
    for category in df[row_kpi_col].unique():
        category_data = df[df[row_kpi_col] == category]['numerical_sel_brate']
        category_mean = category_data.mean()
        between_variance += len(category_data) * (category_mean - df['numerical_sel_brate'].mean())**2

    eta_squared = between_variance / (len(df) * overall_variance)
    return np.sqrt(eta_squared)


#### Helper Functions for Plotting ####
def calculate_frequency_matrix(df, row_kpi_col, col_kpi_col):
    freq_matrix = pd.crosstab(df[row_kpi_col], df[col_kpi_col])
    sorted_columns = sort_bitrate_strings(freq_matrix.columns)
    freq_matrix = freq_matrix.reindex(columns=sorted_columns, fill_value=0)
    total = freq_matrix.values.sum()
    percentage_matrix = (freq_matrix / total) * 100
    return percentage_matrix

def annotate_heatmap(im, data, textcolors=("black", "white"), threshold=10, fontsize=8, **textkw):
    kw = dict(horizontalalignment="center", verticalalignment="center")
    kw.update(textkw)
    for (j, k), val in np.ndenumerate(data):
        if val >= 1.0:
            color = textcolors[int(val > threshold)]
            im.axes.text(k, j, f'{val:.1f}', color=color, fontsize=fontsize, **kw)

def setup_heatmap_axes(ax, title, xlabel, ylabel, xticklabels, yticklabels):
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(np.arange(len(xticklabels)))
    ax.set_yticks(np.arange(len(yticklabels)))
    ax.set_xticklabels(xticklabels, rotation=45, ha='right')
    ax.set_yticklabels(yticklabels)

def plot_heatmap_core(ax, freq_matrix, title, kpi_name, sorted_ytick_labels):
    im = ax.imshow(freq_matrix, cmap='YlOrRd', origin='lower', vmin=0, vmax=20)
    cbar = ax.figure.colorbar(im, ax=ax)
    cbar.set_label('Percentage (%)', rotation=270, labelpad=15)
    setup_heatmap_axes(ax, title, 'Selected Bitrate', f'{kpi_name.capitalize()} State', freq_matrix.columns, sorted_ytick_labels)
    annotate_heatmap(im, freq_matrix.values)

def plot_kpi_heatmaps_historical_revised(history_df, kpi_name, plot_folder_path, full_plot_title, also_show=True):
    """
    Plots heatmaps for historical KPIs and saves the figure, including MI, Cramer's V, and Correlation Ratio.
    """
    for i in range(8):
        history_df[f'{kpi_name}_{i}'] = history_df[f'array_{kpi_name}'].apply(lambda x: x[i])

    symbolic_strings = create_symbolic_strings(kpi_name)
    sorted_symbolic_strings = sort_symbolic_kpi_strings(symbolic_strings)

    fig, axes = plt.subplots(2, 4, figsize=(30, 20)) # Updated to 2 rows, as we only need 8 plots and 4 columns
    axes = axes.flatten()
    fig.suptitle(full_plot_title, fontsize=16, y=1.02)

    for i in range(8):
        ax = axes[i]
        freq_matrix = calculate_frequency_matrix(history_df, f'{kpi_name}_{i}', 'sel_brate')
        freq_matrix = freq_matrix.reindex(index=sorted_symbolic_strings, fill_value=0)
        mi_value = mutual_info_score(history_df[f'{kpi_name}_{i}'], history_df['sel_brate'])
        cramers_v_value = calculate_cramers_v(freq_matrix)
        # Calculate Correlation Ratio, using 'sel_brate' column with numerical values extracted
        correlation_ratio_value = calculate_correlation_ratio(history_df, f'{kpi_name}_{i}', 'sel_brate')

        title = f'{kpi_name.capitalize()}_{i} vs Selected Bitrate\nMI: {mi_value:.2f}, Cramer\'s V: {cramers_v_value:.2f}, Eta: {correlation_ratio_value:.2f}'
        plot_heatmap_core(ax, freq_matrix, title, kpi_name, sorted_symbolic_strings)
        # Drop the temporary numerical column after each plot
        history_df.drop(columns=['numerical_sel_brate'], inplace=True, errors='ignore')


    plt.tight_layout()
    _save_and_show_plot(plt, plot_folder_path, also_show, full_plot_title)


def plot_kpi_heatmaps_revised(symbolic_df, kpi_name, plot_folder_path, full_plot_title, also_show=True):
    """
    Plots heatmaps for non-historical KPIs and saves the figure, including MI, Cramer's V, and Correlation Ratio.
    """
    freq_matrix = calculate_frequency_matrix(symbolic_df, kpi_name, 'sel_brate')
    mi_value = mutual_info_score(symbolic_df[kpi_name], symbolic_df['sel_brate'])
    cramers_v_value = calculate_cramers_v(freq_matrix)
    # Calculate Correlation Ratio, using 'sel_brate' column with numerical values extracted
    correlation_ratio_value = calculate_correlation_ratio(symbolic_df, kpi_name, 'sel_brate')

    unique_kpi_values = sorted(symbolic_df[kpi_name].unique(), key=sort_symbolic_kpi_strings_key)
    freq_matrix = freq_matrix.reindex(index=unique_kpi_values, fill_value=0)
    sorted_ytick_labels = freq_matrix.index

    fig_width = 8 + freq_matrix.shape[1] * 0.5
    fig_height = 6 + freq_matrix.shape[0] * 0.5
    fig, ax = plt.subplots(figsize=(fig_width, fig_height)) # <---- Look here
    fig.suptitle(full_plot_title, fontsize=16, y=1.02)

    title = f'{kpi_name.capitalize()} vs Selected Bitrate\nMI: {mi_value:.2f}, Cramer\'s V: {cramers_v_value:.2f}, Eta: {correlation_ratio_value:.2f}'
    plot_heatmap_core(ax, freq_matrix, title, kpi_name, sorted_ytick_labels)

    plt.tight_layout()
    _save_and_show_plot(plt, plot_folder_path, also_show, full_plot_title)
    # Drop the temporary numerical column after each plot
    symbolic_df.drop(columns=['numerical_sel_brate'], inplace=True, errors='ignore')


def _save_and_show_plot(plt_obj, plot_folder_path, also_show, full_plot_title):
    """
    Saves the plot to a file and optionally shows it.
    """
    if plot_folder_path:
        timestamp = datetime.datetime.now().strftime("%B-%d %H:%M")
        sanitized_title = full_plot_title.replace(" ", "_") if full_plot_title else "no_title"
        sanitized_title = re.sub(r'[^\w\-_.()]', '', sanitized_title)
        base_filename = f"{sanitized_title}_{timestamp}.png"
        timestamped_save_path = os.path.join(plot_folder_path, base_filename)

        os.makedirs(plot_folder_path, exist_ok=True)
        plt_obj.savefig(timestamped_save_path, bbox_inches='tight', dpi=300)
        print(f"Figure saved to: {timestamped_save_path}")

    if also_show:
        plt_obj.show()
    else:
        plt_obj.close()

def create_symbolic_strings(kpi_name):
    predicates = ['dec', 'const', 'inc']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    symbolic_strings = [f"{predicate}({kpi_name}, {category})" for category in categories for predicate in predicates]
    return symbolic_strings

def sort_symbolic_kpi_strings_key(symbolic_string):
    level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
    for level in level_order:
        if level in symbolic_string:
            return level_order.index(level)
    return len(level_order)

def sort_symbolic_kpi_strings(symbolic_strings):
    return sorted(symbolic_strings, key=sort_symbolic_kpi_strings_key, reverse=True)

def create_history_df(symbolic_df, window_size=8, kpis=['dl_tput', 'dl_delay', 'bwidth']): # Added 'bwidth' here
    required_columns = kpis + ['Timestep', 'sel_brate', 'File_name']
    missing_columns = [col for col in required_columns if col not in symbolic_df.columns]
    if missing_columns:
        raise ValueError(f"Input DataFrame is missing required columns: {missing_columns}")
    arrays = {kpi: [None] * window_size for kpi in kpis}
    records = []
    for index, row in symbolic_df.iterrows():
        for kpi in arrays.keys():
            arrays[kpi] = arrays[kpi][1:] + [row[kpi]]
        record = {
            'timestep': row['Timestep'],
            **{f'array_{kpi}': arrays[kpi].copy() for kpi in kpis},
            'sel_brate': row['sel_brate'],
            'File Name': row['File_name']
        }
        records.append(record)
    history_df = pd.DataFrame(records)
    filter_condition = pd.Series(True, index=history_df.index)
    for kpi in kpis:
        filter_condition &= history_df[f'array_{kpi}'].apply(lambda x: None not in x)
    history_df = history_df[filter_condition]
    return history_df

def sort_bitrate_strings(bitrate_strings):
    return sorted(bitrate_strings, key=lambda x: float(re.findall(r'[\d.]+', x)[-1]))


# Example Usage
plot_folder = "outputs/heatmaps_all_metrics" # Define plot output folder
history_df = create_history_df(symbolic_df, kpis=['dl_tput', 'dl_delay', 'bwidth']) # Added 'bwidth' to kpis list


for kpi in ['dl_tput', 'dl_delay', 'bwidth']: # Added 'bwidth' here
    full_plot_title = f"{kpi.capitalize()} Heatmaps - Historical - Metrics"
    plot_kpi_heatmaps_historical_revised(
        history_df,
        kpi,
        plot_folder_path=plot_folder,
        full_plot_title=full_plot_title,
        also_show=False
    )

for kpi in ['dl_tput_all', 'dl_delay_all', 'bwidth_all', 'last_brate', 'buffer', 'rem_chunks']: # Added 'bwidth_all' here
    full_plot_title = f"{kpi.capitalize()} Heatmap - Metrics"
    plot_kpi_heatmaps_revised(
        symbolic_df,
        kpi,
        plot_folder_path=plot_folder,
        full_plot_title=full_plot_title,
        also_show=False
    )

In [ ]:
### Train knowleddge graph on train dataset
rt_kg_test = {
    'last_brate': KnowledgeGraph(["last_brate"]),
    "buffer": KnowledgeGraph(["buffer"]),
    "dl_tput": KnowledgeGraph(["dl_tput"]),
    "dl_delay": KnowledgeGraph(["dl_delay"]),
    "rem_chunks": KnowledgeGraph(["rem_chunks"]),
    # "combined": KnowledgeGraph(["buffer", "dl_tput", "dl_delay"]),
}

for index, row in train_symbolic_df.iterrows():
    
    for key in rt_kg_test:
        rt_kg_test[key].update_graph(row.to_dict())

for key in rt_kg_test:
        rt_kg_test[key]._update_probabilities_and_sizes()

In [ ]:
# Create a list to store all the rows
analysis_rows = []

for index, row in symbolic_df.iterrows():
    timestep = row['Timestep']
    file_name = row['File_name']
    
    for key in rt_kg_test:
        current_state, state_id = rt_kg_test[key]._extract_state(row.to_dict())
        if state_id in rt_kg_test[key].G.nodes:
            outgoing_edges = rt_kg_test[key].G.out_edges(state_id, data=True)
            
            # Collect all actions across all outgoing edges
            all_actions = {}
            for _, _, edge_data in outgoing_edges:
                actions = edge_data.get('actions', {})
                for action, action_data in actions.items():
                    if action not in all_actions:
                        all_actions[action] = {
                            'count': 0,
                            'total_reward': 0
                        }
                    all_actions[action]['count'] += action_data.get('count', 0)
                    all_actions[action]['total_reward'] += (
                        action_data.get('count', 0) * action_data.get('mean_reward', 0)
                    )
            
            # Calculate statistics for each action
            total_count = sum(data['count'] for data in all_actions.values())
            action_stats = {}
            for action, data in all_actions.items():
                prob = data['count'] / total_count if total_count > 0 else 0
                mean_reward = (data['total_reward'] / data['count']) if data['count'] > 0 else 0
                action_stats[action] = {
                    'probability': prob,
                    'mean_reward': mean_reward
                }
            
            # Find best reward and most probable actions
            best_reward_action = max(action_stats.items(), key=lambda x: x[1]['mean_reward'])
            most_probable_action = max(action_stats.items(), key=lambda x: x[1]['probability'])
            
            # Get the action taken by the agent
            taken_action = row['sel_brate']
            
            # Determine what type of choice the agent made
            choice_type = []
            if taken_action == best_reward_action[0]:
                choice_type.append('best_reward')
            if taken_action == most_probable_action[0]:
                choice_type.append('most_probable')
            
            choice_type = '/'.join(choice_type) if choice_type else 'neither'
            
            # Add row to our list
            analysis_rows.append({
                'timestep': timestep,
                'kpi': key,
                'current_state': state_id,
                'taken_action': taken_action,
                'taken_action_reward': action_stats.get(taken_action, {}).get('mean_reward', 0),
                'best_reward_action': best_reward_action[0],
                'best_reward': best_reward_action[1]['mean_reward'],
                'most_probable_action': most_probable_action[0],
                'highest_probability': most_probable_action[1]['probability'],
                'choice_type': choice_type,
                'file_name': file_name,
            })

# Create the dataframe from the collected rows
analysis_df = pd.DataFrame(analysis_rows)

# Print summary statistics
print("\nAnalysis Summary:")
print("\nChoice Type Distribution:")
print(analysis_df['choice_type'].value_counts(normalize=True))

print("\nChoice Type Distribution by KPI:")
print(pd.crosstab(analysis_df['kpi'], analysis_df['choice_type'], normalize='index'))

In [ ]:
def create_symbolic_strings(kpi_name: str) -> List[str]:
    """
    Creates an array of symbolic strings for a given KPI name.
    
    Args:
        kpi_name (str): Name of the KPI
        
    Returns:
        List[str]: List of symbolic strings with all predicate-category combinations
    """
    predicates = ['dec', 'const', 'inc']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    
    symbolic_strings = []
    for category in categories:
        for predicate in predicates:
            symbolic_strings.append(f"{predicate}({kpi_name}, {category})")
    
    return symbolic_strings

def plot_state_choice_correlation(
    analysis_df: pd.DataFrame,
    save_path: Optional[str] = None,
    plot_title: Optional[str] = None,
    also_show: bool = True
) -> None:
    """
    Creates heatmaps showing the correlation between KPI states and choice types.
    
    Args:
        analysis_df (pd.DataFrame): DataFrame containing the analysis data
        save_path (Optional[str]): Path to save the plot
        plot_title (Optional[str]): Title for the plot
        also_show (bool): Whether to display the plot
    """
    # Get unique KPIs
    kpis = analysis_df['kpi'].unique()
    n_kpis = len(kpis)
    
    # Create subplot grid
    fig, axes = plt.subplots(
        (n_kpis + 1) // 2, 
        2, 
        figsize=(20, 6 * ((n_kpis + 1) // 2))
    )
    axes = axes.flatten() if n_kpis > 1 else [axes]
    
    # Add main title if provided
    if plot_title:
        fig.suptitle(plot_title, fontsize=16, y=1.02)
    
    for idx, kpi in enumerate(kpis):
        # Filter data for current KPI
        kpi_data = analysis_df[analysis_df['kpi'] == kpi]
        
        # Get ordered symbolic strings for the current KPI
        symbolic_strings = create_symbolic_strings(kpi)
        # Reverse the order to have VeryHigh at the top
        symbolic_strings = symbolic_strings[::-1]
        
        # Create frequency matrix
        freq_matrix = pd.crosstab(
            kpi_data['current_state'],
            kpi_data['choice_type'],
            normalize='all'
        ) * 100
        
        # Reindex with symbolic strings to ensure consistent ordering
        freq_matrix = freq_matrix.reindex(index=symbolic_strings, fill_value=0)
        
        # Plot heatmap
        im = sns.heatmap(
            freq_matrix,
            ax=axes[idx],
            cmap='YlOrRd',
            vmin=0,
            vmax=freq_matrix.values.max(),
            annot=True,
            fmt='.1f',
            cbar_kws={'label': 'Percentage (%)'}
        )
        
        # Customize plot
        axes[idx].set_title(f'{kpi.upper()} States vs Choice Types')
        axes[idx].set_xlabel('Choice Type')
        axes[idx].set_ylabel('Current State')
        
        # Rotate x-axis labels for better readability
        axes[idx].set_xticklabels(
            axes[idx].get_xticklabels(),
            rotation=45,
            ha='right'
        )
        
        # Rotate y-axis labels for better readability
        axes[idx].set_yticklabels(
            axes[idx].get_yticklabels(),
            rotation=0
        )
    
    # Remove empty subplots if any
    if n_kpis < len(axes):
        for idx in range(n_kpis, len(axes)):
            fig.delaxes(axes[idx])
    
    plt.tight_layout()
    
    # Save plot if path provided
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Figure saved to: {save_path}")
    
    if also_show:
        plt.show()
    else:
        plt.close()

# Example usage
plot_state_choice_correlation(
    analysis_df,
    save_path="outputs/state_choice_correlation.png",
    plot_title="KPI States vs Choice Types Correlation",
    also_show=True
)

## Graph Visualization

### General Stats per kpi

In [ ]:
import matplotlib.pyplot as plt

# Assuming rt_kg is already defined and populated
kpis = ['buffer', 'dl_tput', 'dl_delay', 'rem_chunks', 'bwidth', 'combined', 'dl_tput_all', 'dl_delay_all', 'bwidth_all']

# Visualization code
all_kpi_data = []

for kpi in kpis:
    g, net = rt_kg[kpi].get_graph()
    
    num_nodes = g.number_of_nodes()
    num_edges = g.number_of_edges()
    density = nx.density(g)
    is_directed = g.is_directed()
    
    if is_directed:
        is_strongly_connected = nx.is_strongly_connected(g)
        is_weakly_connected = nx.is_weakly_connected(g)
    else:
        is_connected = nx.is_connected(g)

    degrees = [deg for _, deg in g.degree()]
    avg_degree = sum(degrees) / len(degrees) if degrees else 0
    min_degree = min(degrees) if degrees else 0
    max_degree = max(degrees) if degrees else 0

    kpi_data = {
        'kpi': kpi,
        'nodes': num_nodes,
        'edges': num_edges,
        'density': density,
        'avg_degree': avg_degree,
        'min_degree': min_degree,
        'max_degree': max_degree,
        'is_directed': is_directed,
    }
    
    if is_directed:
        kpi_data['strongly_connected'] = is_strongly_connected
        kpi_data['weakly_connected'] = is_weakly_connected
    else:
        kpi_data['connected'] = is_connected

    all_kpi_data.append(kpi_data)
    net.save_graph(f"./graphs/{kpi}_network_graph.html")


df = pd.DataFrame(all_kpi_data)


# 1. Bar Chart: Nodes and Edges
fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.35
x = np.arange(len(df))

bars1 = ax.bar(x - bar_width/2, df['nodes'], bar_width, label='Nodes', color = 'skyblue')
bars2 = ax.bar(x + bar_width/2, df['edges'], bar_width, label='Edges', color = 'coral')

# Add numbers on top of the bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height,
            f'{int(height)}', ha='center', va='bottom')

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height,
            f'{int(height)}', ha='center', va='bottom')

ax.set_xlabel('KPI', fontsize = 12)
ax.set_ylabel('Count', fontsize = 12)
ax.set_title('Number of Nodes and Edges per KPI', fontsize = 14)
ax.set_xticks(x)
ax.set_xticklabels(df['kpi'], fontsize=10)
ax.legend(fontsize = 10)
ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
plt.tight_layout()
plt.show()


# 2. Bar Chart: Average, Min, Max Degree
fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.2
x = np.arange(len(df))

bars1 = ax.bar(x - bar_width, df['avg_degree'], bar_width, label='Average Degree', color = 'lightgreen')
bars2 = ax.bar(x, df['min_degree'], bar_width, label='Min Degree', color = 'lightcoral')
bars3 = ax.bar(x + bar_width, df['max_degree'], bar_width, label='Max Degree', color = 'lightblue')

# Add numbers on top of the bars
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height,
            f'{height:.2f}', ha='center', va='bottom')

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height,
            f'{height:.2f}', ha='center', va='bottom')

for bar in bars3:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height,
            f'{height:.2f}', ha='center', va='bottom')

ax.set_xlabel('KPI', fontsize = 12)
ax.set_ylabel('Degree', fontsize = 12)
ax.set_title('Degree Statistics per KPI', fontsize = 14)
ax.set_xticks(x)
ax.set_xticklabels(df['kpi'], fontsize=10)
ax.legend(fontsize = 10)
ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
plt.tight_layout()
plt.show()

# 3. Bar Chart: Graph Density
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(df['kpi'], df['density'], color='lightpink')

# Add numbers on top of the bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height,
            f'{height:.2f}', ha='center', va='bottom')

ax.set_xlabel('KPI', fontsize = 12)
ax.set_ylabel('Density', fontsize = 12)
ax.set_title('Graph Density per KPI', fontsize = 14)
ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
plt.tight_layout()
plt.show()

# 4. Connectivity Table (Using matplotlib table for now)
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

table_data = []

for index, row in df.iterrows():
    row_data = [row['kpi']]
    if row['is_directed']:
       row_data.append(row['strongly_connected'])
       row_data.append(row['weakly_connected'])
    else:
        row_data.append(row['connected'])

    table_data.append(row_data)
col_labels = ['KPI']
if df['is_directed'].any():
    col_labels.extend(['Strongly Connected', 'Weakly Connected'])
else:
    col_labels.append('Connected')


table = ax.table(cellText=table_data,
                 colLabels=col_labels,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
plt.tight_layout()
plt.show()

# 5. Scatter plot: Density vs. Average Degree
plt.figure(figsize=(8, 6))
plt.scatter(df['density'], df['avg_degree'], c='purple', s = 100)

for i, txt in enumerate(df['kpi']):
    plt.annotate(txt, (df['density'][i], df['avg_degree'][i]), textcoords="offset points", xytext=(5,5), ha='left')

plt.xlabel('Graph Density', fontsize = 12)
plt.ylabel('Average Degree', fontsize = 12)
plt.title('Graph Density vs. Average Degree', fontsize = 14)
plt.grid(True, linestyle = '--', alpha = 0.7)
plt.tight_layout()
plt.show()

### Graph node probability distribution

In this visualizatoin we want to show that for each node in the graph what was the probability of each of the all possible actions that could be taken.

Train the graph on the train dataset

In [ ]:
rt_kg_test = {
    'last_brate': KnowledgeGraph(["last_brate"]),
    "buffer": KnowledgeGraph(["buffer"]),
    "dl_tput": KnowledgeGraph(["dl_tput"]),
    "dl_delay": KnowledgeGraph(["dl_delay"]),
    'bwidth': KnowledgeGraph(["bwidth"]),
    "rem_chunks": KnowledgeGraph(["rem_chunks"]),
    "dl_tput_all": KnowledgeGraph(["dl_tput_all"]),
    "dl_delay_all": KnowledgeGraph(["dl_delay_all"]),
    'bwidth_all': KnowledgeGraph(["bwidth_all"]),
    # "combined": KnowledgeGraph(["buffer", "dl_tput", "dl_delay"]),
}

for index, row in train_symbolic_df.iterrows():
    
    for key in rt_kg_test:
        rt_kg_test[key].update_graph(row.to_dict())

for key in rt_kg_test:
        rt_kg_test[key]._update_probabilities_and_sizes()

Now We have to write a code that plots the probability distribution of all the nodes in the graph for their action

In [ ]:
from collections import defaultdict

def create_bitrate_symbolic_strings():
    """
    Creates an array of symbolic strings for bitrate actions with all possible combinations,
    sorted with dec before const before inc within each bitrate group.
    """
    VIDEO_BIT_RATE = [300., 750., 1200., 1850., 2850., 4300.]

    symbolic_strings = []

    # Add constant bitrate strings
    for rate in VIDEO_BIT_RATE:
        symbolic_strings.append(f"const(sel_brate, {rate:.1f})")

    # Add increasing/decreasing bitrate strings
    for start_rate in VIDEO_BIT_RATE:
        for final_rate in VIDEO_BIT_RATE:
            if start_rate != final_rate:
                # Determine if it's increasing or decreasing based on bitrate values
                predicate = 'inc' if final_rate > start_rate else 'dec'
                symbolic_strings.append(f"{predicate}(sel_brate, {start_rate:.1f}, {final_rate:.1f})")

    def sort_key_within_group(s):
        if s.startswith('dec'):
            return 0
        elif s.startswith('const'):
            return 1
        elif s.startswith('inc'):
            return 2
        return 3  # Should not happen

    grouped_strings = {}
    for s in symbolic_strings:
        match = re.findall(r'[\d.]+', s)
        if 'const' in s:
            group_key = float(match[0])
        else:
            group_key = float(match[-1])

        if group_key not in grouped_strings:
            grouped_strings[group_key] = []
        grouped_strings[group_key].append(s)

    sorted_strings = []
    for bitrate in sorted(grouped_strings.keys()):
        grouped_strings[bitrate].sort(key=sort_key_within_group)
        sorted_strings.extend(grouped_strings[bitrate])

    return sorted_strings

def create_symbolic_strings(kpi_name):
        """
        Creates an array of symbolic strings for a given KPI name with all possible combinations
        of predicates and categories.
        """
        predicates = ['dec', 'const', 'inc']
        categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']

        symbolic_strings = []
        for category in categories:
            for predicate in predicates:
                symbolic_strings.append(f"{predicate}({kpi_name}, {category})")

        return symbolic_strings

def calculate_action_probabilities(kg, reference_actions):
    """
    Calculate the probability of actions for each node in the knowledge graph.
    
    Args:
    kg (NetworkX.Graph): The knowledge graph
    reference_actions (List[str]): List of all possible actions
    
    Returns:
    Dict[str, Dict[str, float]]: A dictionary of node names to action probabilities
    """
    node_probabilities = {}
    
    for node in kg.G.nodes:
        combined_actions = defaultdict(int)
        total_count = 0
        
        for _, successor, edge_data in kg.G.out_edges(node, data=True):
            actions = edge_data.get('actions', {})
            for action, data in actions.items():
                count = data.get('count', 0)
                combined_actions[action] += count
                total_count += count
        
        probabilities = {}
        for action in reference_actions:
            probabilities[action] = combined_actions[action] / total_count if total_count > 0 else 0
        
        node_probabilities[node] = probabilities
    
    return node_probabilities

def plot_action_probabilities(node_probabilities, reference_actions, kpi_name):
    """
    Plot the action probabilities for each node in the graph with sorted legend and expanded hover text.
    """
    fig = go.Figure()
    
    # Define the order of categories
    category_order = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    
    def get_category_sort_key(node_name):
        # Find which category appears in the node name
        for i, category in enumerate(category_order):
            if category in node_name:
                return i
        return len(category_order)  # Put items without category at the end
    
    # Sort nodes based on categories
    sorted_nodes = sorted(node_probabilities.keys(), key=get_category_sort_key)
    
    for node in sorted_nodes:
        probabilities = node_probabilities[node]
        y_values = [probabilities[action] for action in reference_actions]
        
        fig.add_trace(go.Scatter(
            x=reference_actions,
            y=y_values,
            mode='lines+markers',
            name=node,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate="<b>Action:</b> %{x}<br>" +
                         "<b>Probability:</b> %{y:.3f}<br>" +
                         "<b>Node:</b> " + node +
                         "<extra></extra>"
        ))
    
    # Rest of your layout code remains the same
    fig.update_layout(
        title=dict(
            text=f'Action Probability Distribution for {kpi_name}',
            font=dict(size=24, family="Arial", color="black", weight="bold"),
            x=0.5
        ),
        xaxis_title=dict(
            text='Bitrate Action',
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        yaxis_title=dict(
            text="Probability",
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        hovermode='x unified',
        hoverlabel=dict(
            bgcolor="white",
            font_size=14,
            font_family="Arial",
            namelength=-1  # This ensures the full text is shown
        ),
        height=600,
        margin=dict(l=100, r=200, t=100, b=100),
        font=dict(size=16, family="Arial"),
        legend=dict(
            font=dict(size=14, family="Arial"),
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.05,
            orientation="v",
            traceorder='normal'
        ),
        yaxis=dict(
            tickfont=dict(size=14, family="Arial"),
            tickangle=0,
            showgrid=True,
            gridcolor='lightgray',
            range=[0, 1]
        ),
        xaxis=dict(
            tickmode='array',
            ticktext=reference_actions,
            tickvals=list(range(len(reference_actions))),
            tickfont=dict(size=14, family="Arial"),
            tickangle=45,
            showgrid=True,
            gridcolor='lightgray'
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',
        paper_bgcolor='white'
    )

    fig.show()

# Update the main execution part
# for key, kg in rt_kg_test.items():
for key, kg in {key: rt_kg_test[key] for key in ['bwidth', 'bwidth_all', 'dl_tput', 'dl_tput_all']}.items():
    print(f"Processing Knowledge Graph for KPI: {key}")
    reference_actions = create_bitrate_symbolic_strings()
    node_probabilities = calculate_action_probabilities(kg, reference_actions)
    plot_action_probabilities(node_probabilities, reference_actions, key)

# Visualize the agent's performance - Clean

## Visualize the numerical KPI per video per KPI

In [ ]:
def plot_kpi_metrics(raw_df, kpi_columns, titles, dataset_name):
    """
    Plots KPI metrics over time for each file in the raw_df dataframe and saves the plot to both PNG and HTML files.

    Parameters:
    raw_df (pd.DataFrame): The dataframe containing the raw data with KPI metrics.
    kpi_columns (list): List of column names in raw_df to be plotted as KPIs.
    titles (list): List of titles corresponding to each KPI for the subplots.
    dataset_name (str): The name of the dataset being visualized, used for saving the plot.
    """
    vis_name = "kpi_metrics_over_time" # Define visualization name

    # Create visualization folder path
    folder_path = create_visualization_folder_path(vis_name, dataset_name)

    # Validate input lengths
    if len(kpi_columns) != len(titles):
        raise ValueError("The length of kpi_columns and titles must be the same.")

    # Get unique file names
    unique_files = raw_df['File_name'].unique()

    # Define a color palette
    color_palette = qualitative.Plotly

    # Create a color map for each unique file
    file_colors = {file: color_palette[i % len(color_palette)] for i, file in enumerate(unique_files)}

    # Determine the number of rows and columns for subplots
    num_kpis = len(kpi_columns)
    rows = (num_kpis + 1) // 2  # Ensure enough rows for all KPIs
    cols = 2  # Fixed to 2 columns

    # Create subplots
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=titles)

    # Add traces for each file and KPI
    for file in unique_files:
        df_file = raw_df[raw_df['File_name'] == file].copy()
        df_file['Relative_Timestep'] = df_file['Timestep'] - df_file['Timestep'].min()
        for i, (kpi, title) in enumerate(zip(kpi_columns, titles)):
            row = (i // cols) + 1
            col = (i % cols) + 1
            show_legend = (kpi == kpi_columns[0])  # Show legend only for the first KPI
            fig.add_trace(
                go.Scatter(
                    x=df_file['Relative_Timestep'],
                    y=df_file[kpi],
                    mode='lines',
                    name=file,
                    legendgroup=file,
                    showlegend=show_legend,
                    line=dict(color=file_colors[file])
                ),
                row=row, col=col
            )

    # Update layout for better visuals with vertical lines on hover and full width
    fig.update_layout(
        height=1500,
        autosize=True,
        title_text="KPI Metrics Over Time",
        hovermode='x unified',
        margin=dict(l=50, r=50, t=100, b=50)
    )

    # Add spikes to all x-axes for hover effects
    for i in range(1, num_kpis + 1):
        fig.update_xaxes(showspikes=True, spikecolor="grey", spikethickness=1, row=(i // cols) + 1, col=(i % cols) + 1)

    # Save the plot to the specified folder in PNG format
    png_filename = os.path.join(folder_path, f"kpi_metrics_over_time_{dataset_name}.png") #include dataset name in file name
    fig.write_image(png_filename)
    print(f"KPI Metrics plot saved as PNG to: {png_filename}")

    # Save the plot to the specified folder in HTML format
    html_filename = os.path.join(folder_path, f"kpi_metrics_over_time_{dataset_name}.html") #include dataset name in file name
    fig.write_html(html_filename)
    print(f"KPI Metrics plot saved as HTML to: {html_filename}")


    # Display the interactive plot
    fig.show()

# Example usage:
# Assuming raw_df is already loaded as a pandas DataFrame and dataset is defined
kpi_columns = ['buffer', 'dl_tput', 'dl_delay', 'bwidth', 'rem_chunks', 'sel_brate', 'reward']
titles = ['Buffer Size', 'Download Throughput', 'Download Delay', 'Bandwidth', 'Remaining Chunks', 'Selected Bitrate', 'Reward']
plot_kpi_metrics(raw_df, kpi_columns, titles, dataset_name=dataset) # Pass dataset name here

## Visualize the Markers and the approximator effect

In [ ]:
def plot_kpis_with_markers(raw_df, markers_df, timestep_col, file_name_col, kpis, dataset_name, specific_name="", marker_kpi_col='kpi', title=None):
    """
    Plots KPIs with markers in an n*1 subplot format and saves the plot to both PNG and HTML files.
    Includes an optional specific_name for the visualization folder.

    Parameters:
        raw_df (pd.DataFrame): The dataframe containing the main KPI data.
        markers_df (pd.DataFrame): The dataframe containing the marker data (e.g., percentiles).
        timestep_col (str): The name of the column representing the timestep in both dataframes.
        file_name_col (str): The name of the column representing the file name in both dataframes.
        kpis (list): A list of KPI column names to plot from raw_df.
        dataset_name (str): The name of the dataset being visualized, used for saving the plot.
        specific_name (str, optional): An optional specific name to include in the folder path. Defaults to "".
        marker_kpi_col (str): The name of the column in markers_df that identifies the KPI (default: 'kpi').
        title (str): The title of the plot. If None, a default title will be used.
    """
    vis_name = "kpi_with_markers_over_time" # Define visualization name

    # Create visualization folder path with specific_name
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    n_kpis = len(kpis)

    # Create a subplot with n_kpis rows and 1 column
    fig = make_subplots(rows=n_kpis, cols=1, subplot_titles=kpis, shared_xaxes=True, vertical_spacing=0.05)

    # Get unique file names and sort them naturally
    file_names = natsorted(raw_df[file_name_col].unique())  # Use natsorted for natural sorting

    # Assign colors using the qualitative color palette
    colors = qualitative.Plotly

    # Create a color mapping for each file_name
    color_mapping = {file_name: colors[i % len(colors)] for i, file_name in enumerate(file_names)}

    # Function to lighten a color (simple approach, can be adjusted)
    def lighten_color(color, factor=0.5):
        return f"rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}, {factor})"

    # Plot each KPI
    for i, kpi in enumerate(kpis):
        for j, file_name in enumerate(file_names):
            # Filter raw_df for the current file_name
            df_filtered = raw_df[raw_df[file_name_col] == file_name]
            color = color_mapping[file_name]  # Use the same color for the same file_name across subplots
            light_color = lighten_color(color, factor=0.5)  # Lighten the color for markers

            # Plot the main KPI trace
            fig.add_trace(
                go.Scatter(
                    x=df_filtered[timestep_col],
                    y=df_filtered[kpi],
                    mode='lines',
                    name=file_name,
                    line=dict(color=color),
                    legendgroup=file_name,  # Group traces by file_name
                    showlegend=(i == 0)  # Show legend only for the first subplot
                ),
                row=i+1, col=1
            )

            # Filter markers_df for the current file_name and KPI
            markers_filtered = markers_df[(markers_df[file_name_col] == file_name) & (markers_df[marker_kpi_col] == kpi)]

            # Plot the marker traces (min, p20, p40, p60, p80, max)
            for marker in ['min', 'p20', 'p40', 'p60', 'p80', 'max']:
                fig.add_trace(
                    go.Scatter(
                        x=markers_filtered[timestep_col],
                        y=markers_filtered[marker],
                        mode='lines',
                        line=dict(color=light_color, dash='dash'),
                        name=f'{file_name} {marker}',
                        legendgroup=file_name,  # Group markers with their main trace
                        showlegend=False  # Hide legend for markers
                    ),
                    row=i+1, col=1
                )

    # Update layout
    fig.update_layout(
        height=300 * n_kpis,
        title_text=title if title else "KPIs Over Time by File Name with Markers",
        title_font=dict(size=20, family="Arial", color="black", weight="bold"),  # Make title bold and centered
        title_x=0.5,  # Center the title
        legend_title="File Name",
        hovermode='x unified'  # Enable vertical hovering across all subplots
    )

    # Save the plot to the specified folder in PNG format
    png_filename = os.path.join(folder_path, f"kpi_with_markers_over_time_{dataset_name}.png") #include dataset name in file name
    fig.write_image(png_filename)
    print(f"KPI with Markers plot saved as PNG to: {png_filename}")

    # Save the plot to the specified folder in HTML format
    html_filename = os.path.join(folder_path, f"kpi_with_markers_over_time_{dataset_name}.html") #include dataset name in file name
    fig.write_html(html_filename)
    print(f"KPI with Markers plot saved as HTML to: {html_filename}")

    # Show the plot
    fig.show()


# Example usage
specific_name_suffix = "tdigest_approximator" # Example specific name, set to "" to skip

plot_kpis_with_markers(
    raw_df=raw_df,
    markers_df=markers_df,
    timestep_col='Timestep',
    file_name_col='File_name',
    kpis=['last_brate', 'buffer', 'dl_tput', 'dl_delay', 'rem_chunks', 'bwidth'],
    marker_kpi_col='kpi',  # Optional, default is 'kpi'
    dataset_name=dataset, #pass dataset name here
    specific_name=specific_name_suffix, # Pass specific_name here
    title="<b>KPI's actual values and corresponding marker's values over time (TDigest algo as approximator)</b>"  # Add a custom title (optional)
)

## Visualize the Symbolic Representation

### Temporal View

In [ ]:
def create_kpi_figure_multi_file(symbolic_df, kpi_to_plot, dataset_name, specific_name="", num_files_to_plot=None):
    """
    Creates a figure for a single KPI, showing symbolic representations over time,
    with improved y-axis tick spacing and trend-based ordering, and saves plots.

    Args:
        symbolic_df (pd.DataFrame): DataFrame with symbolic data.
        kpi_to_plot (str): KPI name to plot.
        dataset_name (str): Dataset name for saving plots.
        specific_name (str, optional): Specific name for folder path.
        num_files_to_plot (int, optional): Number of files to plot.

    Returns:
        go.Figure: The created figure object.
    """
    vis_name = "symbolic_kpi_over_time"  # Visualization name

    # Create visualization folder path with specific_name
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    # Get unique symbolic values and sort them
    unique_values = symbolic_df[kpi_to_plot].unique()
    sorted_values = sorted(
        unique_values,
        key=lambda x: (
            {'VeryHigh': 5, 'High': 4, 'Medium': 3, 'Low': 2, 'VeryLow': 1}.get(re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x).group(), 0),
            {'spiking': 7, 'fluctuating': 5, 'dropping': 3}.get(re.search(r'(spiking|fluctuating|dropping)', x).group() if re.search(r'(spiking|fluctuating|dropping)', x) else 'fluctuating', 2), # Trend ordering
            {'inc': 3, 'const': 2, 'dec': 1}.get(re.search(r'(inc|const|dec)', x).group() if re.search(r'(inc|const|dec)', x) else 'const', 2) # Predicate ordering
        )
    )

    # Create value to position mapping with increased spacing and trend consideration
    current_pos = 0
    value_to_position = {}
    last_category = None

    for value in sorted_values:
        category = re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', value).group() if re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', value) else None

        if last_category is None or category != last_category:
            if last_category is not None:
                current_pos += 4  # Increased spacing between categories
            last_category = category

        value_to_position[value] = current_pos
        current_pos += 3  # Increased spacing WITHIN each category group (changed from 2 to 3)


    # Create the plot
    fig = go.Figure()
    colors = qualitative.Plotly

    # Get unique file names and limit if needed
    unique_files = natsorted(symbolic_df['File_name'].unique())
    if num_files_to_plot is not None:
        unique_files = unique_files[:num_files_to_plot]

    # Add traces for each file
    for file_idx, file in enumerate(unique_files):
        file_df = symbolic_df[symbolic_df['File_name'] == file].copy()
        file_df['Relative_Timestep'] = file_df['Timestep'] - file_df['Timestep'].min()
        file_df['y_position'] = file_df[kpi_to_plot].map(value_to_position)

        fig.add_trace(go.Scatter(
            x=file_df['Relative_Timestep'],
            y=file_df['y_position'],
            mode='lines',
            name=file,
            line=dict(color=colors[file_idx % len(colors)], width=2)
        ))

    # Update layout
    fig.update_layout(
        title=dict(
            text=f'{kpi_to_plot} over Time',
            font=dict(size=24, family="Arial", color="black", weight="bold"),
            x=0.5
        ),
        xaxis_title=dict(
            text='Timestep',
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        yaxis_title=dict(
            text="Symbolic Value",
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        hovermode='x unified',
        height=800,
        # Increased left margin here to give more space to y-axis labels
        margin=dict(l=200, r=200, t=100, b=100),
        font=dict(size=16, family="Arial"), # Increased base font size
        legend=dict(
            font=dict(size=14, family="Arial"),
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.05,
            orientation="v"
        ),
        yaxis=dict(
            tickmode='array',
            ticktext=[v for v in sorted_values],
            tickvals=[value_to_position[v] for v in sorted_values],
            tickfont=dict(size=14),  # Increased y-axis tick font size
            tickangle=0,
            showgrid=True,
            gridcolor='lightgray',
            automargin=True  # Add this to automatically adjust margin
        ),
        xaxis=dict(
            tickfont=dict(size=16, family="Arial"),
            showgrid=True,
            gridcolor='lightgray'
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',
        paper_bgcolor='white'
    )

    # Save plots
    png_filename = os.path.join(folder_path, f"symbolic_kpi_over_time_{kpi_to_plot}_{dataset_name}.png")
    fig.write_image(png_filename)
    print(f"Symbolic KPI plot saved as PNG to: {png_filename}")

    html_filename = os.path.join(folder_path, f"symbolic_kpi_over_time_{kpi_to_plot}_{dataset_name}.html")
    fig.write_html(html_filename)
    print(f"Symbolic KPI plot saved as HTML to: {html_filename}")

    return fig


# Example usage
kpi_columns_to_plot = ['buffer', 'dl_tput', 'dl_delay', 'rem_chunks', 'bwidth', 'bwidth_all', 'dl_delay_all', 'dl_tput_all']
num_files_to_plot = 12
specific_name_suffix = ""  # Updated specific name

for kpi in kpi_columns_to_plot:
    fig = create_kpi_figure_multi_file(symbolic_df, kpi, dataset_name=dataset, specific_name=specific_name_suffix, num_files_to_plot=num_files_to_plot)
    fig.show()

### Probabilistic View

In [ ]:
def create_probability_distribution_figure(symbolic_df, kpi_to_plot, dataset_name, specific_name=""):
    """
    Creates a figure showing the probability distribution of symbolic values for a given KPI,
    grouped by file name, using a line plot, and saves the plot to both PNG and HTML files.

    Args:
        symbolic_df (pd.DataFrame): The input DataFrame with symbolic data.
        kpi_to_plot (str): The KPI name to plot.
        dataset_name (str): The name of the dataset being visualized, used for saving the plot.
        specific_name (str, optional): An optional specific name to include in the folder path. Defaults to "".

    Returns:
        go.Figure: The created figure object
    """
    vis_name = "symbolic_rep_probability_distribution" # Define visualization name

    # Create visualization folder path with specific_name
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    # Get unique values and sort them according to our criteria
    unique_values = symbolic_df[kpi_to_plot].unique()
    sorted_values = sorted(
        unique_values,
        key=lambda x: (
            {'VeryHigh': 5, 'High': 4, 'Medium': 3, 'Low': 2, 'VeryLow': 1}.get(re.search(r'(VeryHigh|High|Medium|Low|VeryLow)', x).group(), 0),
            {'inc': 3, 'const': 2, 'dec': 1}.get(re.search(r'(inc|const|dec)', x).group() if re.search(r'(inc|const|dec)', x) else 'const', 2)
        )
    )

    # Create value to position mapping
    value_to_position = {value: idx for idx, value in enumerate(sorted_values)}

    # Create the plot
    fig = go.Figure()
    colors = qualitative.Plotly  # Use Plotly's qualitative color palette

    # Get unique file names and sort them naturally
    unique_files = natsorted(symbolic_df['File_name'].unique())

    # Add traces for each file
    for file_idx, file in enumerate(unique_files):
        file_df = symbolic_df[symbolic_df['File_name'] == file].copy()
        # Calculate the probability distribution
        value_counts = file_df[kpi_to_plot].value_counts(normalize=True)
        probabilities = [value_counts.get(value, 0) for value in sorted_values]

        fig.add_trace(go.Scatter(
            x=[value_to_position[v] for v in sorted_values],
            y=probabilities,
            mode='lines+markers',
            name=file,
            line=dict(color=colors[file_idx % len(colors)], width=2),
            marker=dict(size=8)
        ))

    # Update layout
    fig.update_layout(
        title=dict(
            text=f'Probability Distribution of {kpi_to_plot}',
            font=dict(size=24, family="Arial", color="black", weight="bold"),
            x=0.5  # Center the title
        ),
        xaxis_title=dict(
            text='Symbolic Value',
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        yaxis_title=dict(
            text="Probability",
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        hovermode='x unified',
        height=600,  # Adjusted height for better readability
        margin=dict(l=100, r=200, t=100, b=100),  # Increased right margin for legend
        font=dict(size=16, family="Arial"),
        legend=dict(
            font=dict(size=14, family="Arial"),
            yanchor="top",
            y=1,  # Position legend at the top-right
            xanchor="left",
            x=1.05,  # Move legend to the right of the plot
            orientation="v"  # Vertical orientation for better readability
        ),
        yaxis=dict(
            tickfont=dict(size=14, family="Arial"),
            tickangle=0,  # No rotation needed
            showgrid=True,  # Add gridlines for better readability
            gridcolor='lightgray',
            range=[0, 1]  # Set y-axis range from 0 to 1
        ),
        xaxis=dict(
            tickmode='array',
            ticktext=sorted_values,
            tickvals=[value_to_position[v] for v in sorted_values],
            tickfont=dict(size=14, family="Arial"),
            tickangle=45,  # Rotate x-axis labels for better readability
            showgrid=True,  # Add gridlines for better readability
            gridcolor='lightgray'
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',  # Light gray background for the plot region
        paper_bgcolor='white'  # White background for the entire plot
    )

    # Save the plot to the specified folder in PNG format
    png_filename = os.path.join(folder_path, f"probability_distribution_{kpi_to_plot}_{dataset_name}.png") #include dataset and kpi name in file name
    fig.write_image(png_filename)
    print(f"Probability Distribution plot saved as PNG to: {png_filename}")

    # Save the plot to the specified folder in HTML format
    html_filename = os.path.join(folder_path, f"probability_distribution_{kpi_to_plot}_{dataset_name}.html") #include dataset and kpi name in file name
    fig.write_html(html_filename)
    print(f"Probability Distribution plot saved as HTML to: {html_filename}")

    return fig

# Plot each KPI in a separate figure
kpi_columns_to_plot = ['buffer', 'dl_tput', 'dl_delay', 'rem_chunks', 'bwidth', 'bwidth_all', 'dl_delay_all', 'dl_tput_all']
specific_name_suffix = "" # Example specific name, set to "" to skip

for kpi in kpi_columns_to_plot:
    fig = create_probability_distribution_figure(symbolic_df, kpi, dataset_name=dataset, specific_name=specific_name_suffix)
    fig.show()

## Analyze MI for all kpis including new representation of the symbolic representation

In [ ]:
#### Helper Functions for Statistical Measures ####
def calculate_cramers_v(confusion_matrix):
    """
    Calculates Cramer's V for a given confusion matrix.
    """
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.values.sum()
    min_dim = min(confusion_matrix.shape) - 1
    if min_dim == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))

def extract_bitrate_value(sel_brate_str):
    """
    Extracts the numerical bitrate value from the symbolic 'sel_brate' string,
    getting the LAST numerical value if multiple are present.
    """
    matches = re.findall(r'[-+]?\d*\.\d+|\d+', sel_brate_str) # Find all numerical values
    if matches:
        return float(matches[-1])  # Return the last matched number as float
    else:
        return np.nan  # Return NaN if no number is found (for robustness)

def calculate_correlation_ratio(df, row_kpi_col, col_kpi_col):
    """
    Calculates the Correlation Ratio (Eta) for categorical row_kpi_col and numerical col_kpi_col.
    """
    # Ensure col_kpi_col is numeric by extracting bitrate values
    df['numerical_sel_brate'] = df[col_kpi_col].apply(extract_bitrate_value)
    overall_variance = df['numerical_sel_brate'].var()
    if overall_variance == 0:
        return 0.0

    between_variance = 0.0
    for category in df[row_kpi_col].unique():
        category_data = df[df[row_kpi_col] == category]['numerical_sel_brate']
        category_mean = category_data.mean()
        between_variance += len(category_data) * (category_mean - df['numerical_sel_brate'].mean())**2

    eta_squared = between_variance / (len(df) * overall_variance)
    return np.sqrt(eta_squared)


#### Helper Functions for Plotting ####
def calculate_frequency_matrix(df, row_kpi_col, col_kpi_col):
    freq_matrix = pd.crosstab(df[row_kpi_col], df[col_kpi_col])
    sorted_columns = sort_bitrate_strings(freq_matrix.columns)
    freq_matrix = freq_matrix.reindex(columns=sorted_columns, fill_value=0)
    total = freq_matrix.values.sum()
    percentage_matrix = (freq_matrix / total) * 100
    return percentage_matrix

def annotate_heatmap(im, data, textcolors=("black", "white"), threshold=10, fontsize=10, **textkw): # Increased fontsize
    kw = dict(horizontalalignment="center", verticalalignment="center")
    kw.update(textkw)
    for (j, k), val in np.ndenumerate(data):
        if val >= 1.0:
            color = textcolors[int(val > threshold)]
            im.axes.text(k, j, f'{val:.1f}', color=color, fontsize=fontsize, **kw)

def setup_heatmap_axes(ax, title, xlabel, ylabel, xticklabels, yticklabels):
    ax.set_title(title, fontsize=14) # Increased title fontsize
    ax.set_xlabel(xlabel, fontsize=12) # Increased xlabel fontsize
    ax.set_ylabel(ylabel, fontsize=12) # Increased ylabel fontsize
    ax.set_xticks(np.arange(len(xticklabels)))
    ax.set_yticks(np.arange(len(yticklabels)))
    ax.set_xticklabels(xticklabels, rotation=45, ha='right', fontsize=10) # Increased tick label fontsize
    ax.set_yticklabels(yticklabels, fontsize=10) # Increased tick label fontsize

def plot_heatmap_core(ax, freq_matrix, title, kpi_name, sorted_ytick_labels):
    im = ax.imshow(freq_matrix, cmap='YlOrRd', origin='lower', vmin=0, vmax=20)
    cbar = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04) # Adjusted colorbar size and padding
    cbar.set_label('Percentage (%)', rotation=270, labelpad=15, fontsize=12) # Increased colorbar label fontsize
    cbar.ax.tick_params(labelsize=10) # Increased colorbar tick fontsize
    setup_heatmap_axes(ax, title, 'Selected Bitrate', f'{kpi_name.capitalize()} State', freq_matrix.columns, sorted_ytick_labels)
    annotate_heatmap(im, freq_matrix.values)

def plot_kpi_heatmaps_historical_revised(history_df, kpi_name, plot_folder_path, full_plot_title, dataset_name, specific_name="", also_show=True): # Added dataset_name and specific_name
    """
    Plots heatmaps for historical KPIs and saves the figure, including MI, Cramer's V, and Correlation Ratio.
    """
    vis_name = "kpi_heatmaps" # Define visualization name

    plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name) # use create_visualization_folder_path

    for i in range(8):
        history_df[f'{kpi_name}_{i}'] = history_df[f'array_{kpi_name}'].apply(lambda x: x[i])

    symbolic_strings = create_symbolic_strings(kpi_name)
    sorted_symbolic_strings = sort_symbolic_kpi_strings(symbolic_strings)

    fig, axes = plt.subplots(2, 4, figsize=(30, 20))
    axes = axes.flatten()
    fig.suptitle(full_plot_title, fontsize=16, y=1.02)

    for i in range(8):
        freq_matrix = calculate_frequency_matrix(history_df, f'{kpi_name}_{i}', 'sel_brate')
        freq_matrix = freq_matrix.reindex(index=sorted_symbolic_strings, fill_value=0)
        mi_value = mutual_info_score(history_df[f'{kpi_name}_{i}'], history_df['sel_brate'])
        cramers_v_value = calculate_cramers_v(freq_matrix)
        # Calculate Correlation Ratio, using 'sel_brate' column with numerical values extracted
        correlation_ratio_value = calculate_correlation_ratio(history_df, f'{kpi_name}_{i}', 'sel_brate')

        ax = axes[i]
        title = f'{kpi_name.capitalize()}_{i} vs Selected Bitrate\nMI: {mi_value:.2f}, Cramer\'s V: {cramers_v_value:.2f}, Eta: {correlation_ratio_value:.2f}'
        plot_heatmap_core(ax, freq_matrix, title, kpi_name, sorted_symbolic_strings)
        # Drop the temporary numerical column after each plot
        history_df.drop(columns=['numerical_sel_brate'], inplace=True, errors='ignore')


    plt.tight_layout()
    _save_and_show_plot(plt, plot_folder_path, also_show, full_plot_title, dataset_name, specific_name, kpi_name, is_historical=True) # Added is_historical flag to _save_and_show_plot

def plot_kpi_heatmaps_revised(symbolic_df, kpi_name, plot_folder_path, full_plot_title, dataset_name, specific_name="", also_show=True): # Added dataset_name and specific_name
    """
    Plots heatmaps for non-historical KPIs and saves the figure, including MI, Cramer's V, and Correlation Ratio.
    """
    vis_name = "kpi_heatmaps" # Define visualization name
    plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name) # use create_visualization_folder_path

    freq_matrix = calculate_frequency_matrix(symbolic_df, kpi_name, 'sel_brate')
    mi_value = mutual_info_score(symbolic_df[kpi_name], symbolic_df['sel_brate'])
    cramers_v_value = calculate_cramers_v(freq_matrix)
    # Calculate Correlation Ratio, using 'sel_brate' column with numerical values extracted
    correlation_ratio_value = calculate_correlation_ratio(symbolic_df, kpi_name, 'sel_brate')

    unique_kpi_values = sorted(symbolic_df[kpi_name].unique(), key=sort_symbolic_kpi_strings_key)
    freq_matrix = freq_matrix.reindex(index=unique_kpi_values, fill_value=0)
    sorted_ytick_labels = freq_matrix.index

    fig_width = 8 + freq_matrix.shape[1] * 0.5
    fig_height = 6 + freq_matrix.shape[0] * 0.5
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    fig.suptitle(full_plot_title, fontsize=16, y=1.02)

    title = f'{kpi_name.capitalize()} vs Selected Bitrate\nMI: {mi_value:.2f}, Cramer\'s V: {cramers_v_value:.2f}, Eta: {correlation_ratio_value:.2f}'
    plot_heatmap_core(ax, freq_matrix, title, kpi_name, sorted_ytick_labels)

    plt.tight_layout()
    _save_and_show_plot(plt, plot_folder_path, also_show, full_plot_title, dataset_name, specific_name, kpi_name, is_historical=False) # Added is_historical=False flag to _save_and_show_plot
    # Drop the temporary numerical column after each plot
    symbolic_df.drop(columns=['numerical_sel_brate'], inplace=True, errors='ignore')


def _save_and_show_plot(plt_obj, plot_folder_path, also_show, full_plot_title, dataset_name, specific_name, kpi_name, is_historical): # added dataset_name and specific_name and kpi_name and is_historical
    """
    Saves the plot to a file and optionally shows it.
    """
    vis_name = "kpi_heatmaps" # Define visualization name

    if plot_folder_path:
        timestamp = datetime.datetime.now().strftime("%B-%d_%H%M") # updated timestamp format
        sanitized_title = full_plot_title.replace(" ", "_") if full_plot_title else "no_title"
        sanitized_title = re.sub(r'[^\w\-_.()]', '', sanitized_title)

        filename_parts = [vis_name, kpi_name, dataset_name] # start with common parts
        if is_historical: # Add "historical" if it's a historical heatmap
            filename_parts.append("historical")
        if specific_name: # Add specific_name if it's not empty
            filename_parts.append(specific_name)

        base_filename = "_".join(part for part in filename_parts if part) + "_" + timestamp + ".png" # simplified filename

        timestamped_save_path = os.path.join(plot_folder_path, base_filename)

        os.makedirs(plot_folder_path, exist_ok=True)
        plt_obj.savefig(timestamped_save_path, bbox_inches='tight', dpi=300)
        print(f"Figure saved to: {timestamped_save_path}")

    if also_show:
        plt_obj.show()
    else:
        plt_obj.close()

def create_visualization_folder_path(vis_name, dataset_name, specific_name=""): # added specific_name
    """
    Creates the folder path for storing visualization figures and ensures the path exists.
    Includes an optional specific_name in the path.

    Args:
        vis_name (str): The name of the visualization type (folder name).
        dataset_name (str): The name of the dataset (subfolder name).
        specific_name (str, optional): An optional specific name to include in the folder path. Defaults to "".

    Returns:
        str: The full path to the visualization folder.
    """
    base_dir = "visualizations"
    folder_path = os.path.join(base_dir, vis_name, dataset_name)
    if specific_name:  # Add specific_name to path if it's not empty
        folder_path = os.path.join(folder_path, specific_name)
    os.makedirs(folder_path, exist_ok=True)  # Create directory if it doesn't exist
    return folder_path


def create_symbolic_strings(kpi_name):
    predicates = ['dec', 'const', 'inc']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    symbolic_strings = [f"{predicate}({kpi_name}, {category})" for category in categories for predicate in predicates]
    return symbolic_strings

def sort_symbolic_kpi_strings_key(symbolic_string):
    level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
    for level in level_order:
        if level in symbolic_string:
            return level_order.index(level)
    return len(level_order)

def sort_symbolic_kpi_strings(symbolic_strings):
    return sorted(symbolic_strings, key=sort_symbolic_kpi_strings_key, reverse=True)

def create_history_df(symbolic_df, window_size=8, kpis=['dl_tput', 'dl_delay']):
    required_columns = kpis + ['Timestep', 'sel_brate', 'File_name']
    missing_columns = [col for col in required_columns if col not in symbolic_df.columns]
    if missing_columns:
        raise ValueError(f"Input DataFrame is missing required columns: {missing_columns}")
    arrays = {kpi: [None] * window_size for kpi in kpis}
    records = []
    for index, row in symbolic_df.iterrows():
        for kpi in arrays.keys():
            arrays[kpi] = arrays[kpi][1:] + [row[kpi]]
        record = {
            'timestep': row['Timestep'],
            **{f'array_{kpi}': arrays[kpi].copy() for kpi in kpis},
            'sel_brate': row['sel_brate'],
            'File Name': row['File_name']
        }
        records.append(record)
    history_df = pd.DataFrame(records)
    filter_condition = pd.Series(True, index=history_df.index)
    for kpi in kpis:
        filter_condition &= history_df[f'array_{kpi}'].apply(lambda x: None not in x)
    history_df = history_df[filter_condition]
    return history_df

def sort_bitrate_strings(bitrate_strings):
    return sorted(bitrate_strings, key=lambda x: float(re.findall(r'[\d.]+', x)[-1]))


# Example Usage
plot_folder = "visualizations/kpi_heatmaps" # Define plot output folder inside visualizations
history_df = create_history_df(symbolic_df, kpis=['dl_tput', 'dl_delay', 'bwidth'])
specific_name_suffix = "" # Example specific name, set to "" to skip


for kpi in ['dl_tput', 'dl_delay', 'bwidth']:
    full_plot_title = f"{kpi.capitalize()} Heatmaps - Historical - Metrics"
    plot_kpi_heatmaps_historical_revised(
        history_df,
        kpi,
        plot_folder_path=plot_folder,
        full_plot_title=full_plot_title,
        dataset_name=dataset, # pass dataset name
        specific_name=specific_name_suffix, # pass specific_name
        also_show=False
    )

for kpi in ['dl_tput_all', 'dl_delay_all', 'bwidth_all', 'last_brate', 'buffer', 'rem_chunks']: # Removed extra KPIs for brevity in example
    full_plot_title = f"{kpi.capitalize()} Heatmap - Metrics"
    plot_kpi_heatmaps_revised(
        symbolic_df,
        kpi,
        plot_folder_path=plot_folder,
        full_plot_title=full_plot_title,
        dataset_name=dataset, # pass dataset name
        specific_name=specific_name_suffix, # pass specific_name
        also_show=False
    )

## Graph Visualization - General Stats per kpi

In [ ]:
def plot_node_edge_bar_chart(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a bar chart of nodes and edges per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_bar_charts"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    fig, ax = plt.subplots(figsize=(14, 8), dpi=150) # Increased figsize and dpi
    bar_width = 0.35
    x = np.arange(len(df))

    bars1 = ax.bar(x - bar_width/2, df['nodes'], bar_width, label='Nodes', color = 'skyblue')
    bars2 = ax.bar(x + bar_width/2, df['edges'], bar_width, label='Edges', color = 'coral')

    # Add numbers on top of the bars
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{int(height)}', ha='center', va='bottom', fontsize=12) # Increased fontsize
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{int(height)}', ha='center', va='bottom', fontsize=12) # Increased fontsize

    ax.set_xlabel('KPI', fontsize = 14) # Increased fontsize
    ax.set_ylabel('Count', fontsize = 14) # Increased fontsize
    ax.set_title('Number of Nodes and Edges per KPI', fontsize = 16) # Increased fontsize
    ax.set_xticks(x)
    ax.set_xticklabels(df['kpi'], fontsize=12) # Increased fontsize
    ax.legend(fontsize = 12) # Increased fontsize
    ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"node_edge_bar_chart_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"node_edge_bar_chart_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(fig) # Close the figure to free memory
    print(f"Node and Edge Bar Chart saved to: {filepath}")


def plot_degree_stats_bar_chart(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a bar chart of degree statistics per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_bar_charts"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists


    fig, ax = plt.subplots(figsize=(14, 8), dpi=150) # Increased figsize and dpi
    bar_width = 0.2
    x = np.arange(len(df))

    bars1 = ax.bar(x - bar_width, df['avg_degree'], bar_width, label='Average Degree', color = 'lightgreen')
    bars2 = ax.bar(x, df['min_degree'], bar_width, label='Min Degree', color = 'lightcoral')
    bars3 = ax.bar(x + bar_width, df['max_degree'], bar_width, label='Max Degree', color = 'lightblue')

    # Add numbers on top of the bars
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize
    for bar in bars3:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize

    ax.set_xlabel('KPI', fontsize = 14) # Increased fontsize
    ax.set_ylabel('Degree', fontsize = 14) # Increased fontsize
    ax.set_title('Degree Statistics per KPI', fontsize = 16) # Increased fontsize
    ax.set_xticks(x)
    ax.set_xticklabels(df['kpi'], fontsize=12) # Increased fontsize
    ax.legend(fontsize = 12) # Increased fontsize
    ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"degree_stats_bar_chart_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"degree_stats_bar_chart_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(fig) # Close the figure to free memory
    print(f"Degree Statistics Bar Chart saved to: {filepath}")

def plot_density_bar_chart(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a bar chart of graph density per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_bar_charts"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150) # Increased figsize and dpi
    bars = ax.bar(df['kpi'], df['density'], color='lightpink')

    # Add numbers on top of the bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center', va='bottom', fontsize=12) # Increased fontsize

    ax.set_xlabel('KPI', fontsize = 14) # Increased fontsize
    ax.set_ylabel('Density', fontsize = 14) # Increased fontsize
    ax.set_title('Graph Density per KPI', fontsize = 16) # Increased fontsize
    ax.grid(axis = 'y', linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"density_bar_chart_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"density_bar_chart_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(fig) # Close the figure to free memory
    print(f"Density Bar Chart saved to: {filepath}")

def plot_connectivity_table(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a table of graph connectivity per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_tables"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    fig, ax = plt.subplots(figsize=(14, 5), dpi=150) # Increased figsize and dpi
    ax.axis('off')

    table_data = []
    for index, row in df.iterrows():
        row_data = [row['kpi']]
        if row['is_directed']:
           row_data.append(str(row['strongly_connected'])) # Convert bool to str for table
           row_data.append(str(row['weakly_connected'])) # Convert bool to str for table
        else:
            row_data.append(str(row['connected'])) # Convert bool to str for table
        table_data.append(row_data)

    col_labels = ['KPI']
    if df['is_directed'].any():
        col_labels.extend(['Strongly Connected', 'Weakly Connected'])
    else:
        col_labels.append('Connected')

    table = ax.table(cellText=table_data,
                     colLabels=col_labels,
                     loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12) # Increased fontsize for table
    table.scale(1.2, 2) # Scale table for better readability
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"connectivity_table_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"connectivity_table_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, bbox_inches='tight', dpi=150) # Use bbox_inches='tight' for tables and ensure DPI is passed
    plt.close(fig) # Close the figure to free memory
    print(f"Connectivity Table saved to: {filepath}")


def plot_density_vs_degree_scatter(df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a scatter plot of density vs. average degree per KPI.

    Args:
        df (pd.DataFrame): DataFrame containing KPI network data.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "network_stats_scatter_plots"
    plot_folder_path = os.path.join(folder_path, vis_name) # Updated path - removed dataset_name from here
    os.makedirs(plot_folder_path, exist_ok=True) # Make sure folder exists

    plt.figure(figsize=(10, 8), dpi=150) # Increased figsize and dpi
    plt.scatter(df['density'], df['avg_degree'], c='purple', s = 150) # Increased marker size

    for i, txt in enumerate(df['kpi']):
        plt.annotate(txt, (df['density'][i], df['avg_degree'][i]), textcoords="offset points", xytext=(5,5), ha='left', fontsize=12) # Increased fontsize

    plt.xlabel('Graph Density', fontsize = 14) # Increased fontsize
    plt.ylabel('Average Degree', fontsize = 14) # Increased fontsize
    plt.title('Graph Density vs. Average Degree', fontsize = 16) # Increased fontsize
    plt.grid(True, linestyle = '--', alpha = 0.7)
    plt.tight_layout()

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"density_vs_degree_scatter_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"density_vs_degree_scatter_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, dpi=150) # Ensure DPI is passed to savefig as well
    plt.close(plt.gcf()) # Close the figure to free memory
    print(f"Density vs. Average Degree Scatter Plot saved to: {filepath}")


def create_visualization_folder_path(vis_name, dataset_name, specific_name=""): # Updated function
    """
    Creates the folder path for storing visualization figures and ensures the path exists.
    Includes an optional specific_name in the path.

    Args:
        vis_name (str): The name of the visualization type (folder name).
        dataset_name (str): The name of the dataset (subfolder name).
        specific_name (str, optional): An optional specific name to include in the folder path. Defaults to "".

    Returns:
        str: The full path to the visualization folder.
    """
    base_dir = "visualizations"
    folder_path = os.path.join(base_dir, "network_graph_general_stats", dataset_name, specific_name, vis_name) # Updated path
    os.makedirs(folder_path, exist_ok=True)  # Create directory if it doesn't exist
    return folder_path


# Assuming rt_kg is already defined and populated
kpis = ['last_brate', 'buffer', 'dl_tput', 'bwidth', 'bwidth_all', 'dl_tput_all', 'dl_delay', 'dl_delay_all', 'rem_chunks', 'combined']
specific_name_suffix = "" # Optional specific name for folder

all_kpi_data = []
network_vis_folder = create_visualization_folder_path("network_graphs", dataset, specific_name_suffix) # Folder for network graphs

for kpi in kpis:
    g, net = rt_kg[kpi].get_graph()

    num_nodes = g.number_of_nodes()
    num_edges = g.number_of_edges()
    density = nx.density(g)
    is_directed = g.is_directed()

    if is_directed:
        is_strongly_connected = nx.is_strongly_connected(g)
        is_weakly_connected = nx.is_weakly_connected(g)
    else:
        is_connected = nx.is_connected(g)

    degrees = [deg for _, deg in g.degree()]
    avg_degree = sum(degrees) / len(degrees) if degrees else 0
    min_degree = min(degrees) if degrees else 0
    max_degree = max(degrees) if degrees else 0

    kpi_data = {
        'kpi': kpi,
        'nodes': num_nodes,
        'edges': num_edges,
        'density': density,
        'avg_degree': avg_degree,
        'min_degree': min_degree,
        'max_degree': max_degree,
        'is_directed': is_directed,
    }

    if is_directed:
        kpi_data['strongly_connected'] = is_strongly_connected
        kpi_data['weakly_connected'] = is_weakly_connected
    else:
        kpi_data['connected'] = is_connected

    all_kpi_data.append(kpi_data)
    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    network_filename = f"{kpi}_network_graph_{dataset}_{specific_name_suffix}_{timestamp}.html" if specific_name_suffix else f"{kpi}_network_graph_{dataset}_{timestamp}.html"
    network_filepath = os.path.join(network_vis_folder, network_filename)
    net.save_graph(network_filepath) # Save network graph in the dedicated folder
    print(f"Network graph for {kpi} saved to: {network_filepath}")


df = pd.DataFrame(all_kpi_data)
folder_path = create_visualization_folder_path("", dataset, specific_name_suffix) # Updated base folder path - vis_name is "" now

# Generate and save plots
plot_node_edge_bar_chart(df, folder_path, dataset, specific_name_suffix)
plot_degree_stats_bar_chart(df, folder_path, dataset, specific_name_suffix)
plot_density_bar_chart(df, folder_path, dataset, specific_name_suffix)
plot_connectivity_table(df, folder_path, dataset, specific_name_suffix)
plot_density_vs_degree_scatter(df, folder_path, dataset, specific_name_suffix)

## Check the repeatability of the 8 timestep sequences

In [ ]:
#### Conver the compact History of buffer and delay into seperate columns so that we can perform analysis on them
def create_history_df(symbolic_df, window_size=8, kpis=['dl_tput', 'dl_delay', 'bwidth']):
    """
    Creates a history DataFrame that tracks the last n timesteps for specified KPIs.

    Args:
        symbolic_df (pd.DataFrame): Input DataFrame containing the symbolic data
        window_size (int): Number of timesteps to track (default: 8)
        kpis (list): List of KPI names to track (default: ['buffer', 'dl_delay'])

    Returns:
        pd.DataFrame: A DataFrame containing the rolling history of KPIs
    """
    # Validate inputs
    required_columns = kpis + ['Timestep', 'sel_brate', 'File_name']
    missing_columns = [col for col in required_columns if col not in symbolic_df.columns]
    if missing_columns:
        raise ValueError(f"Input DataFrame is missing required columns: {missing_columns}")

    # Initialize the arrays to keep track of the last n timesteps
    arrays = {kpi: [None] * window_size for kpi in kpis}

    # Initialize an empty list to store the records
    records = []

    # Iterate through the symbolic_df dataframe
    for index, row in symbolic_df.iterrows():
        # Update the arrays with the current timestep's values
        for kpi in arrays.keys():
            arrays[kpi] = arrays[kpi][1:] + [row[kpi]]

        # Create a record for the current timestep
        record = {
            'timestep': row['Timestep'],
            **{f'array_{kpi}': arrays[kpi].copy() for kpi in kpis},  # Store copies of all KPI arrays
            'sel_brate': row['sel_brate'],
            'File Name': row['File_name']
        }

        # Append the record to the list
        records.append(record)

    # Create a new dataframe from the records
    history_df = pd.DataFrame(records)

    # Filter out rows where any of the KPI history values are None
    filter_condition = pd.Series(True, index=history_df.index)
    for kpi in kpis:
        filter_condition &= history_df[f'array_{kpi}'].apply(lambda x: None not in x)

    history_df = history_df[filter_condition]

    return history_df

history_df = create_history_df(symbolic_df)

history_df['array_dl_tput'] = history_df['array_dl_tput'].apply(tuple)
history_df['array_dl_delay'] = history_df['array_dl_delay'].apply(tuple)
history_df['array_bwidth'] = history_df['array_bwidth'].apply(tuple)

In [ ]:
history_df['array_dl_tput'].value_counts()

In [ ]:
history_df['array_dl_delay'].value_counts()

In [ ]:
history_df['array_bwidth'].value_counts()

## Graph node probability distribution

In this visualizatoin we want to show that for each node in the graph what was the probability of each of the all possible actions that could be taken.

In [ ]:
rt_kg_test = {
    'last_brate': KnowledgeGraph(["last_brate"]),
    "buffer": KnowledgeGraph(["buffer"]),
    "dl_tput": KnowledgeGraph(["dl_tput"]),
    "dl_delay": KnowledgeGraph(["dl_delay"]),
    'bwidth': KnowledgeGraph(["bwidth"]),
    "rem_chunks": KnowledgeGraph(["rem_chunks"]),
    'dl_delay_all': KnowledgeGraph(["dl_delay_all"]),
    'dl_tput_all': KnowledgeGraph(["dl_tput_all"]),
    'bwidth_all': KnowledgeGraph(["bwidth_all"]),
    # "combined": KnowledgeGraph(["buffer", "dl_tput", "dl_delay"]),
}

for index, row in train_symbolic_df.iterrows():
    
    for key in rt_kg_test:
        rt_kg_test[key].update_graph(row.to_dict())

for key in rt_kg_test:
        rt_kg_test[key]._update_probabilities_and_sizes()

In [ ]:
def create_bitrate_symbolic_strings():
    """
    Creates an array of symbolic strings for bitrate actions with all possible combinations.
    """
    symbolic_strings = []

    # Add constant bitrate strings
    for rate in VIDEO_BIT_RATE:
        symbolic_strings.append(f"const(sel_brate, {rate:.1f})")

    # Add increasing/decreasing bitrate strings
    for start_rate in VIDEO_BIT_RATE:
        for final_rate in VIDEO_BIT_RATE:
            if start_rate != final_rate:
                # Determine if it's increasing or decreasing based on bitrate values
                predicate = 'inc' if final_rate > start_rate else 'dec'
                symbolic_strings.append(f"{predicate}(sel_brate, {start_rate:.1f}, {final_rate:.1f})")

    def sort_key_within_group(s):
        if s.startswith('dec'):
            return 0
        elif s.startswith('const'):
            return 1
        elif s.startswith('inc'):
            return 2
        return 3  # Should not happen

    grouped_strings = {}
    for s in symbolic_strings:
        match = re.findall(r'[\d.]+', s)
        if 'const' in s:
            group_key = float(match[0])
        else:
            group_key = float(match[-1])

        if group_key not in grouped_strings:
            grouped_strings[group_key] = []
        grouped_strings[group_key].append(s)

    sorted_strings = []
    for bitrate in sorted(grouped_strings.keys()):
        grouped_strings[bitrate].sort(key=sort_key_within_group)
        sorted_strings.extend(grouped_strings[bitrate])

    return sorted_strings

def create_symbolic_strings_kpi(kpi_name): # Renamed to avoid conflict with bitrate function
    """
    Creates an array of symbolic strings for a given KPI name with all possible combinations
    of predicates and categories.
    """
    predicates = ['dec', 'const', 'inc']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']

    symbolic_strings = []
    for category in categories:
        for predicate in predicates:
            symbolic_strings.append(f"{predicate}({kpi_name}, {category})")

    return symbolic_strings

def calculate_action_probabilities(kg, reference_actions):
    """
    Calculate the probability of actions for each node in the knowledge graph.

    Args:
    kg (KnowledgeGraph): The knowledge graph object
    reference_actions (List[str]): List of all possible actions

    Returns:
    Dict[str, Dict[str, float]]: A dictionary of node names to action probabilities
    """
    node_probabilities = {}

    for node in kg.G.nodes:
        combined_actions = defaultdict(int)
        total_count = 0

        for _, successor, edge_data in kg.G.out_edges(node, data=True):
            actions = edge_data.get('actions', {})
            for action, data in actions.items():
                count = data.get('count', 0)
                combined_actions[action] += count
                total_count += count

        probabilities = {}
        for action in reference_actions:
            probabilities[action] = combined_actions[action] / total_count if total_count > 0 else 0

        node_probabilities[node] = probabilities

    return node_probabilities

def plot_kpi_action_probabilities(rt_kg_test, kpi_name, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves action probability distribution plot for a given KPI.

    Args:
        rt_kg_test (dict): Dictionary of KnowledgeGraph objects.
        kpi_name (str): Name of the KPI to plot for.
        folder_path (str): Base folder path to save visualizations.
        dataset_name (str): Dataset name for folder structure.
        specific_name (str, optional): Specific name for folder path.
    """
    vis_name = "action_probability_distributions"
    plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
    reference_actions = create_bitrate_symbolic_strings()
    node_probabilities = calculate_action_probabilities(rt_kg_test[kpi_name], reference_actions)
    _plot_action_probabilities_plotly(node_probabilities, reference_actions, kpi_name, plot_folder_path, dataset_name, specific_name)


def _plot_action_probabilities_plotly(node_probabilities, reference_actions, kpi_name, plot_folder_path, dataset_name, specific_name=""):
    """
    Helper function to create and save the Plotly action probability plot.
    """
    fig = go.Figure()

    sorted_node_names = create_symbolic_strings_kpi(kpi_name) # Use KPI specific symbolic strings
    sorted_nodes = sorted(node_probabilities.keys(),
                         key=lambda x: sorted_node_names.index(x) if x in sorted_node_names else float('inf'))

    for node in sorted_nodes:
        probabilities = node_probabilities[node]
        y_values = [probabilities[action] for action in reference_actions]

        fig.add_trace(go.Scatter(
            x=reference_actions,
            y=y_values,
            mode='lines+markers',
            name=node,
            line=dict(width=2),
            marker=dict(size=8),
            hovertemplate="<b>Action:</b> %{x}<br>" +
                          "<b>Probability:</b> %{y:.3f}<br>" +
                          "<b>State:</b> " + node + # Updated hover text to "State"
                          "<extra></extra>"  # This removes the secondary box
        ))

    fig.update_layout(
        title=dict(
            text=f'Action Probabilities for {kpi_name} KPI <br> (Dataset: {dataset_name})', # Updated title with dataset and clarity
            font=dict(size=24, family="Arial", color="black", weight="bold"),
            x=0.5
        ),
        xaxis_title=dict(
            text='Bitrate Action',
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        yaxis_title=dict(
            text="Probability",
            font=dict(size=18, family="Arial", color="black", weight="bold")
        ),
        hovermode='x unified',
        hoverlabel=dict(
            bgcolor="white",
            font_size=14,
            font_family="Arial",
            namelength=-1  # This ensures the full text is shown
        ),
        height=600,
        margin=dict(l=100, r=200, t=100, b=100),
        font=dict(size=16, family="Arial"),
        legend=dict(
            font=dict(size=14, family="Arial"),
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.05,
            orientation="v",
            traceorder='normal',
            title="KPI State" # Updated Legend title to "KPI State"
        ),
        yaxis=dict(
            tickfont=dict(size=14, family="Arial"),
            tickangle=0,
            showgrid=True,
            gridcolor='lightgray',
            range=[0, 1]
        ),
        xaxis=dict(
            tickmode='array',
            ticktext=reference_actions,
            tickvals=list(range(len(reference_actions))),
            tickfont=dict(size=14, family="Arial"),
            tickangle=45,
            showgrid=True,
            gridcolor='lightgray'
        ),
        plot_bgcolor='rgba(240, 240, 240, 0.9)',
        paper_bgcolor='white'
    )

    timestamp = datetime.datetime.now().strftime("%H%M") # Shortened timestamp format
    filename_base = f"action_prob_{kpi_name}_{dataset_name}_{specific_name}_{timestamp}" if specific_name else f"action_prob_{kpi_name}_{dataset_name}_{timestamp}" # Optimized filename
    png_filepath = os.path.join(plot_folder_path, filename_base + ".png")
    html_filepath = os.path.join(plot_folder_path, filename_base + ".html")

    fig.write_image(png_filepath, scale=2) # Save as PNG, scale for higher res
    fig.write_html(html_filepath) # Save as HTML

    print(f"Action Probability Distribution plot for {kpi_name} saved to: {png_filepath} and {html_filepath}")


# Assuming rt_kg_test and dataset are defined from previous notebook cells
dataset_name = dataset  # Use dataset from notebook, e.g., 'car'
specific_name_suffix = "action_prob_dist" # Example specific name
folder_path = "visualizations" # Base visualization folder path


kpis_to_plot = ['last_brate', 'buffer', 'dl_tput', 'dl_tput_all', 'dl_delay', 'dl_delay_all', 'bwidth', 'bwidth_all', 'rem_chunks'] # Example KPIs to plot

for kpi in kpis_to_plot:
    plot_kpi_action_probabilities(rt_kg_test, kpi, folder_path, dataset_name, specific_name_suffix)

## calculate the number of times the action taken by agent was the most rewarding or the most probable

In this visualization we want to see if for a graph trained in train dataset, then how many times those actions were made on the test dataset.

In [ ]:
### Train knowleddge graph on train dataset
rt_kg_test = {
    'last_brate': KnowledgeGraph(["last_brate"]),
    "buffer": KnowledgeGraph(["buffer"]),
    "dl_tput": KnowledgeGraph(["dl_tput"]),
    "dl_delay": KnowledgeGraph(["dl_delay"]),
    'bwidth': KnowledgeGraph(["bwidth"]),
    "rem_chunks": KnowledgeGraph(["rem_chunks"]),
    'dl_delay_all': KnowledgeGraph(["dl_delay_all"]),
    'dl_tput_all': KnowledgeGraph(["dl_tput_all"]),
    'bwidth_all': KnowledgeGraph(["bwidth_all"]),
    # "combined": KnowledgeGraph(["buffer", "dl_tput", "dl_delay"]),
}

for index, row in train_symbolic_df.iterrows():
    
    for key in rt_kg_test:
        rt_kg_test[key].update_graph(row.to_dict())

for key in rt_kg_test:
        rt_kg_test[key]._update_probabilities_and_sizes()

# Create a list to store all the rows
analysis_rows = []

for index, row in symbolic_df.iterrows():
    timestep = row['Timestep']
    file_name = row['File_name']
    
    for key in rt_kg_test:
        current_state, state_id = rt_kg_test[key]._extract_state(row.to_dict())
        if state_id in rt_kg_test[key].G.nodes:
            outgoing_edges = rt_kg_test[key].G.out_edges(state_id, data=True)
            
            # Collect all actions across all outgoing edges
            all_actions = {}
            for _, _, edge_data in outgoing_edges:
                actions = edge_data.get('actions', {})
                for action, action_data in actions.items():
                    if action not in all_actions:
                        all_actions[action] = {
                            'count': 0,
                            'total_reward': 0
                        }
                    all_actions[action]['count'] += action_data.get('count', 0)
                    all_actions[action]['total_reward'] += (
                        action_data.get('count', 0) * action_data.get('mean_reward', 0)
                    )
            
            # Calculate statistics for each action
            total_count = sum(data['count'] for data in all_actions.values())
            action_stats = {}
            for action, data in all_actions.items():
                prob = data['count'] / total_count if total_count > 0 else 0
                mean_reward = (data['total_reward'] / data['count']) if data['count'] > 0 else 0
                action_stats[action] = {
                    'probability': prob,
                    'mean_reward': mean_reward
                }
            
            # Find best reward and most probable actions
            best_reward_action = max(action_stats.items(), key=lambda x: x[1]['mean_reward'])
            most_probable_action = max(action_stats.items(), key=lambda x: x[1]['probability'])
            
            # Get the action taken by the agent
            taken_action = row['sel_brate']
            
            # Determine what type of choice the agent made
            choice_type = []
            if taken_action == best_reward_action[0]:
                choice_type.append('best_reward')
            if taken_action == most_probable_action[0]:
                choice_type.append('most_probable')
            
            choice_type = '/'.join(choice_type) if choice_type else 'neither'
            
            # Add row to our list
            analysis_rows.append({
                'timestep': timestep,
                'kpi': key,
                'current_state': state_id,
                'taken_action': taken_action,
                'taken_action_reward': action_stats.get(taken_action, {}).get('mean_reward', 0),
                'best_reward_action': best_reward_action[0],
                'best_reward': best_reward_action[1]['mean_reward'],
                'most_probable_action': most_probable_action[0],
                'highest_probability': most_probable_action[1]['probability'],
                'choice_type': choice_type,
                'file_name': file_name,
            })

# Create the dataframe from the collected rows
analysis_df = pd.DataFrame(analysis_rows)

# # Print summary statistics
# print("\nAnalysis Summary:")
# print("\nChoice Type Distribution:")
# print(analysis_df['choice_type'].value_counts(normalize=True))

# print("\nChoice Type Distribution by KPI:")
# print(pd.crosstab(analysis_df['kpi'], analysis_df['choice_type'], normalize='index'))

In [ ]:
def create_symbolic_strings_kpi(kpi_name: str, analysis_df: pd.DataFrame) -> List[str]:
    """
    Dynamically creates and sorts symbolic strings for a KPI, ensuring "VeryHigh" is at the top.
    """
    unique_states = list(analysis_df[analysis_df['kpi'] == kpi_name]['current_state'].unique())

    def sort_key_symbolic_state(state_str):
        """Sort key to order symbolic states with VeryHigh at the top."""
        level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
        for level in level_order:
            if level in state_str:
                return level_order.index(level)
        return len(level_order)  # Default to bottom if no level found

    sorted_states = sorted(unique_states, key=sort_key_symbolic_state) # sort by category
    return sorted_states[::-1] # reverse to put VeryHigh at top

def _plot_state_choice_heatmap_core(ax, freq_matrix, kpi_name):
    """
    Helper function to plot a single heatmap for KPI state vs choice type.
    """
    sns.heatmap(
        freq_matrix,
        ax=ax,
        cmap='YlOrRd',
        vmin=0,
        vmax=freq_matrix.values.max(),
        annot=True,
        fmt='.1f',
        cbar_kws={'label': 'Percentage (%)'}
    )

    ax.set_title(f'{kpi_name.upper()} States vs Choice Types')
    ax.set_xlabel('Choice Type')
    ax.set_ylabel('Current State')

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=45,
        ha='right'
    )

    ax.set_yticklabels(
        ax.get_yticklabels(),
        rotation=0,
        verticalalignment='center'
    )


def plot_kpi_state_choice_correlation_heatmaps(analysis_df, folder_path, dataset_name, specific_name=""):
    """
    Creates and saves heatmaps showing the correlation between KPI states and choice types.
    """
    vis_name = "state_choice_heatmaps"
    plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    kpis = analysis_df['kpi'].unique()
    n_kpis = len(kpis)

    fig, axes = plt.subplots(
        (n_kpis + 1) // 2,
        2,
        figsize=(20, 6 * ((n_kpis + 1) // 2)),
        dpi=150
    )
    axes = axes.flatten() if n_kpis > 1 else [axes]
    fig.suptitle(f'KPI States vs Choice Types Correlation (Dataset: {dataset_name})', fontsize=16, y=0.95)

    for idx, kpi in enumerate(kpis):
        kpi_data = analysis_df[analysis_df['kpi'] == kpi]
        symbolic_strings = create_symbolic_strings_kpi(kpi, kpi_data) # Pass kpi_data for unique states

        freq_matrix = pd.crosstab(
            kpi_data['current_state'],
            kpi_data['choice_type'],
            normalize='all'
        ) * 100
        freq_matrix = freq_matrix.reindex(index=symbolic_strings, fill_value=0)

        _plot_state_choice_heatmap_core(axes[idx], freq_matrix, kpi)


    if n_kpis < len(axes):
        for idx in range(n_kpis, len(axes)):
            fig.delaxes(axes[idx])

    plt.tight_layout(rect=[0, 0, 1, 0.93])

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"state_choice_correlation_heatmaps_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"state_choice_correlation_heatmaps_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, bbox_inches='tight', dpi=150)
    plt.close(fig)
    print(f"KPI State vs Choice Type Heatmaps saved to: {filepath}")


def plot_kpi_choice_type_distribution_bar_chart(analysis_df, folder_path, dataset_name, specific_name=""):
    """
    Generates and saves a bar chart of choice type distribution per KPI.
    """
    vis_name = "choice_type_distribution_bar_charts"
    plot_folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    choice_type_distribution_kpi = pd.crosstab(analysis_df['kpi'], analysis_df['choice_type'], normalize='index')

    fig, ax = plt.subplots(figsize=(14, 8), dpi=150)

    bars = choice_type_distribution_kpi.plot(kind='bar', stacked=True, ax=ax, colormap='viridis')

    ax.set_title(f'Choice Type Distribution by KPI (Dataset: {dataset_name})', fontsize=16)
    ax.set_xlabel('KPI', fontsize=14)
    ax.set_ylabel('Proportion', fontsize=14)
    ax.tick_params(axis='x', rotation=45, labelsize=12)
    ax.tick_params(axis='y', labelsize=12)
    ax.legend(title='Choice Type', fontsize=12, bbox_to_anchor=(1.05, 1), loc='upper left') # Legend outside

    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Add percentage labels inside each bar section
    for bars_container in ax.containers:
        ax.bar_label(bars_container, fmt='%.1f%%', label_type='center', fontsize=10, color='black')

    plt.tight_layout(rect=[0, 0, 0.9, 1])

    timestamp = datetime.datetime.now().strftime("%B-%d_%H%M")
    filename = f"choice_type_distribution_bar_chart_{dataset_name}_{specific_name}_{timestamp}.png" if specific_name else f"choice_type_distribution_bar_chart_{dataset_name}_{timestamp}.png"
    filepath = os.path.join(plot_folder_path, filename)
    plt.savefig(filepath, bbox_inches='tight', dpi=150)
    plt.close(fig)
    print(f"Choice Type Distribution Bar Chart saved to: {filepath}")


# Assuming analysis_df and dataset are defined from previous notebook cells
dataset_name = dataset # e.g., 'car'
specific_name_suffix = "choice_type_analysis" # Example specific name
folder_path = "visualizations" # Base visualization folder path

plot_kpi_state_choice_correlation_heatmaps(analysis_df, folder_path, dataset_name, specific_name_suffix)
plot_kpi_choice_type_distribution_bar_chart(analysis_df, folder_path, dataset_name, specific_name_suffix)

## KPI influence calculation

In this section we wnat to see wether we can calculate the influence of each kpi on agent's action or not.

Train the graph on training dataset

The graph doesn't contain any information about the KPI states in the test dataset.

In [ ]:
rt_kg_test = {
    'last_brate': KnowledgeGraph(["last_brate"]),
    "buffer": KnowledgeGraph(["buffer"]),
    "dl_tput": KnowledgeGraph(["dl_tput"]),
    "dl_delay": KnowledgeGraph(["dl_delay"]),
    'bwidth': KnowledgeGraph(["bwidth"]),
    "rem_chunks": KnowledgeGraph(["rem_chunks"]),
    'dl_delay_all': KnowledgeGraph(["dl_delay_all"]),
    'dl_tput_all': KnowledgeGraph(["dl_tput_all"]),
    'bwidth_all': KnowledgeGraph(["bwidth_all"]),
}

kpis_list_test = list(rt_kg_test.keys()) # Create a list of kpi names

for index, row in train_symbolic_df.iterrows():
    for key in rt_kg_test:
        rt_kg_test[key].update_graph(row.to_dict())

for key in rt_kg_test:
    rt_kg_test[key]._update_probabilities_and_sizes()

### IS with delta function in (binary, decaying and hierarchical) + representation

In [ ]:
def kl_divergence(p_dist, q_dist):
    """
    Calculates the Kullback-Leibler divergence D_KL(P || Q).
    """
    kl_div = 0.0
    actions_p = set(p_dist.keys())
    actions_q = set(q_dist.keys())
    all_actions = actions_p.union(actions_q)

    for action in all_actions:
        p_prob = p_dist.get(action, 0.0)
        q_prob = q_dist.get(action, 0.0)

        if p_prob > 0.0:
            if q_prob <= 0.0:
                continue
            else:
                kl_div += p_prob * np.log(p_prob / q_prob)

    return kl_div

def get_recommended_action(action_distribution):
    """
    Returns the action with the highest probability from the action distribution.
    """
    if not action_distribution:
        return None
    return max(action_distribution, key=action_distribution.get)

def binary_delta_function(recommended_action, actual_action):
    """
    Implements the binary delta function.
    """
    if recommended_action is None or actual_action is None:
        return 0.0
    if recommended_action == actual_action:
        return 1
    else:
        return 0

def extract_bitrate_change(action_str):
    """
    Extracts bitrate change information from FOL action strings
    (const, inc, dec).

    Returns start_bitrate, end_bitrate as floats, or None, None if parsing fails.
    For 'const', start_bitrate and end_bitrate will be the same constant bitrate.
    """
    const_match = re.match(r'const\(sel_brate, ([-+]?\d*\.\d+|\d+)\)', action_str)
    inc_dec_match = re.match(r'(inc|dec)\(sel_brate, ([-+]?\d*\.\d+|\d+), ([-+]?\d*\.\d+|\d+)\)', action_str)

    if const_match:
        bitrate = float(const_match.group(1))
        return bitrate, bitrate  # For const, start and end are the same
    elif inc_dec_match:
        start_bitrate = float(inc_dec_match.group(2))
        end_bitrate = float(inc_dec_match.group(3))
        return start_bitrate, end_bitrate
    else:
        return None, None  # Parsing failed

def decaying_delta_function(recommended_action, actual_action, sigma=0.5):
    """
    Implements the decaying delta function.
    """
    if recommended_action is None or actual_action is None:
        return 0.0  # Or some other appropriate default value

    rec_start_br, rec_end_br = extract_bitrate_change(recommended_action)
    act_start_br, act_end_br = extract_bitrate_change(actual_action)

    if rec_end_br is None or act_end_br is None:
        return 0.0
    max_delta = act_end_br
    if max_delta == 0:
        return 0.0

    distance = abs(act_end_br - rec_end_br) / max_delta
    delta_value = np.exp(-(distance**2) / (2 * sigma**2))
    return delta_value

def hierarchical_delta_function(recommended_action, actual_action):
    """
    Implements the hierarchical multi-resolution delta function.
    """
    if recommended_action is None or actual_action is None:
      return 0.0

    rec_start_br, rec_end_br = extract_bitrate_change(recommended_action)
    act_start_br, act_end_br = extract_bitrate_change(actual_action)

    if rec_end_br is None or act_end_br is None:
        return 0.0

    # Direction Match (mathbb{I}(dir(a_t) = dir(a_k^*)))
    rec_action_type = recommended_action.split('(')[0] # e.g., 'inc', 'dec', 'const'
    act_action_type = actual_action.split('(')[0]

    if rec_action_type == act_action_type: # Directions match
        direction_match = 1
    else:
        direction_match = 0

    # Magnitude Similarity (1 - (|Δ_t - Δ_k^*| / maxΔ))
    max_delta = act_end_br
    if max_delta == 0: # Avoid division by zero
        magnitude_similarity = 0.0
    else:
        magnitude_similarity = 1 - (abs(act_end_br - rec_end_br) / max_delta)
        magnitude_similarity = max(0, magnitude_similarity) # Ensure it's not negative

    delta_value = direction_match * magnitude_similarity
    return delta_value


def divergence_weighted_influence(rt_kg, kpis, row, agent_action, delta_function_type='binary', sigma_decay=0.5):
    """
    Calculates the divergence-weighted influence for each KPI, allowing different delta functions.
    """
    all_P_k = {}
    for kpi_name in kpis:
        kpi_value = getattr(row, kpi_name)
        kg = rt_kg[kpi_name]

        edges = kg.G.out_edges(kpi_value, data=True)
        P_k = defaultdict(float)
        total_occurrence = sum(edge[2]['occurrence'] for edge in edges)
        if total_occurrence == 0:
            print(f"Warning: No outgoing edges from state {kpi_value} for KPI {kpi_name}. P_k(a) will be empty.")
        else:
            for _, _, edge_data in edges:
                for action, action_data in edge_data['actions'].items():
                    P_k[action] += (action_data['count'] / total_occurrence)
        all_P_k[kpi_name] = P_k
    
    # print(f"P_K Distribution:")
    # pprint(all_P_k)
    
    # Calculate Baseline Distribution P_emptyset(a)
    P_emptyset = defaultdict(float)
    num_kpis = len(kpis)
    all_actions = set()

    for kpi_name in kpis:
        P_k = all_P_k[kpi_name]
        all_actions.update(P_k.keys())

    for action in all_actions:
        P_emptyset[action] = 0.0

    for kpi_name in kpis:
        P_k = all_P_k[kpi_name]
        for action, prob in P_k.items():
            P_emptyset[action] += prob

    for action in P_emptyset:
        P_emptyset[action] /= num_kpis
    
    # print(f"Baseline Distribution P_emptyset(a):")
    # pprint(P_emptyset)
    
    influence_scores = {}
    kl_divergences = {}
    delta_values = {}
    
    for kpi_name in kpis:
        # print(f"\n\n Calculating Influence Scores for KPI: {kpi_name}")
        P_k = all_P_k[kpi_name]
        kl_div_value = kl_divergence(P_k, P_emptyset)
        kl_divergences[kpi_name] = kl_div_value
        
        recommended_action_kpi = get_recommended_action(P_k)
        
        # print(f"KPI: {kpi_name} ({getattr(row, kpi_name)})")
        # pprint(f"P_K Distribution:")
        # pprint(P_k)
        # pprint(f"Recommended Action: {recommended_action_kpi}")
        
        delta_value = 0 # Initialize delta_value
        
        # print(f"\n--> Agent Action: {agent_action}")
        # print(f"\n--> Recommended Action: {recommended_action_kpi}")
                
        if delta_function_type == 'binary':
            # print(f"\n--> Delta Function Type: Binary")
            delta_value = binary_delta_function(recommended_action_kpi, agent_action)
        elif delta_function_type == 'decaying':
            # print(f"\n--> Delta Function Type: Decaying with Sigma: {sigma_decay}")
            delta_value = decaying_delta_function(recommended_action_kpi, agent_action, sigma_decay)
            # print(f"Delta Value: {delta_value}")
            # print(f"influence score is {kl_div_value * delta_value}")
        elif delta_function_type == 'hierarchical':
            # print(f"\n--> Delta Function Type: Hierarchical")
            delta_value = hierarchical_delta_function(recommended_action_kpi, agent_action)
            # print(f"Delta Value: {delta_value}")
            # print(f"influence score is {kl_div_value * delta_value}")
        else:
            raise ValueError(f"Unknown delta_function_type: {delta_function_type}")

        delta_values[kpi_name] = delta_value
        influence_score = kl_div_value * delta_value
        influence_scores[kpi_name] = influence_score

    return influence_scores, kl_divergences, delta_values


influence_scores_list_binary = [] # List for binary delta influence scores
influence_scores_list_decaying = [] # List for decaying delta influence scores
influence_scores_list_hierarchical = [] # List for hierarchical delta influence scores


for index, row in symbolic_df.iterrows():
    timestep = row['Timestep']
    agent_action = getattr(row, 'sel_brate')
    kpi_values = {kpi_name: getattr(row, kpi_name) for kpi_name in kpis_list_test}

    # print(f"\n{'='*50}")
    # print(f"Timestep: {timestep}")
    # print(f"Agent Action: {agent_action}")
    # print(f"\nKPI Symbolic Values:")
    # for kpi_name, kpi_value in kpi_values.items():
    #     print(f"  {kpi_name}: {kpi_value}")
    # print("\n")


    # Calculate influence with Binary Delta Function
    influence_score_binary, kl_divergences_binary, delta_values_binary = divergence_weighted_influence(rt_kg_test, kpis_list_test, row, agent_action, delta_function_type='binary')
    
    influence_scores_list_binary.append({
        "Timestep": row['Timestep'],
        **influence_score_binary,
        "File_name": row['File_name'],
        "delta_type": "binary"
    })

    # Calculate influence with Decaying Delta Function
    influence_score_decaying, kl_divergences_decaying, delta_values_decaying = divergence_weighted_influence(rt_kg_test, kpis_list_test, row, agent_action, delta_function_type='decaying', sigma_decay=0.5)
    influence_scores_list_decaying.append({
        "Timestep": row['Timestep'],
        **influence_score_decaying,
        "File_name": row['File_name'],
        "delta_type": "decaying"
    })

    # Calculate influence with Hierarchical Delta Function
    influence_score_hierarchical, kl_divergences_hierarchical, delta_values_hierarchical = divergence_weighted_influence(rt_kg_test, kpis_list_test, row, agent_action, delta_function_type='hierarchical')

    influence_scores_list_hierarchical.append({
        "Timestep": row['Timestep'],
        **influence_score_hierarchical,
        "File_name": row['File_name'],
        "delta_type": "hierarchical"
    })
    
    # if index > 10:
    #     break
    
    for key in rt_kg_test:
        rt_kg_test[key].update_graph(row.to_dict())
        rt_kg_test[key]._update_probabilities_and_sizes()

    # print(f"\n{'='*50}")


IS_df_binary = pd.DataFrame(influence_scores_list_binary)
IS_df_decaying = pd.DataFrame(influence_scores_list_decaying)
IS_df_hierarchical = pd.DataFrame(influence_scores_list_hierarchical)

# print("\n\nInfluence Scores DataFrame - Binary Delta (IS_df_binary):")
# print(IS_df_binary)
# print("\nInfluence Scores DataFrame - Decaying Delta (IS_df_decaying):")
# print(IS_df_decaying)
# print("\nInfluence Scores DataFrame - Hierarchical Delta (IS_df_hierarchical):")
# print(IS_df_hierarchical)

-----------------------------------

### Visualize - numerical values

In [ ]:
def plot_kpis_raw_and_multiple_is_improved(raw_df, is_dfs, is_styles, timestep_col, file_name_col, kpis, dataset_name, specific_name="", title=None):
    """
    Plots KPIs from raw_df and multiple IS_df with improved layout.
    """
    vis_name = "kpi_raw_and_multiple_is_improved"  # Updated vis_name
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    n_kpis = len(kpis)
    # Reduce vertical_spacing and add row/col specs for secondary y-axis
    fig = make_subplots(rows=n_kpis, cols=1,
                        subplot_titles=[f"<b>{kpi}</b>" for kpi in kpis],
                        shared_xaxes=True, vertical_spacing=0.02,  # Reduced spacing
                        specs=[[{'secondary_y': True}] for _ in range(n_kpis)])

    file_names = natsorted(raw_df[file_name_col].unique())
    colors = qualitative.Plotly
    color_mapping = {file_name: colors[i % len(colors)] for i, file_name in enumerate(file_names)}

    for i, kpi in enumerate(kpis):
        if kpi.endswith('_all'):
            def get_kpi_value(row, kpi_name):
                if isinstance(row[kpi_name], (list, np.ndarray)):
                    if kpi_name == 'bwidth_all':
                        return row[kpi_name][3] if len(row[kpi_name]) > 3 else None
                    else:
                        return row[kpi_name][-1] if len(row[kpi_name]) > 0 else None
                else:
                    return row[kpi_name]
        else:
            def get_kpi_value(row, kpi_name):
                return row[kpi_name]

        for j, file_name in enumerate(file_names):
            raw_df_filtered = raw_df[raw_df[file_name_col] == file_name]
            color = color_mapping[file_name]

            fig.add_trace(
                go.Scatter(
                    x=raw_df_filtered[timestep_col],
                    y=raw_df_filtered.apply(lambda row: get_kpi_value(row, kpi), axis=1),
                    mode='lines',
                    name=f"{file_name} (Raw)",  # Simplified name
                    line=dict(color=color),
                    legendgroup=file_name,
                    showlegend=(i == 0)
                ),
                row=i + 1, col=1, secondary_y=False
            )

            for is_df, style in zip(is_dfs, is_styles):
                is_df_type = is_df['delta_type'].iloc[0]
                is_df_filtered = is_df[is_df[file_name_col] == file_name]
                fig.add_trace(
                    go.Scatter(
                        x=is_df_filtered[timestep_col],
                        y=is_df_filtered[kpi],
                        mode='lines',
                        name=f"{file_name} (IS - {is_df_type}, {style})",  # Simplified style
                        line=dict(color=color, dash=style),
                        legendgroup=file_name,
                        showlegend=(i == 0)
                    ),
                    row=i + 1, col=1, secondary_y=True
                )

        # Y-Axis Updates (Larger Font, Automargin)
        fig.update_yaxes(
            title_text=f"<b>Raw KPI Value</b>",
            secondary_y=False,
            row=i + 1, col=1,
            title_font=dict(size=16),  # Larger title font
            tickfont=dict(size=14),   # Larger tick label font
            automargin=True  # Important for label visibility
        )
        fig.update_yaxes(
            title_text=f"<b>Influence Score</b>",
            secondary_y=True,
            row=i + 1, col=1,
            title_font=dict(size=16),
            tickfont=dict(size=14),
            automargin=True
        )

    # Overall Layout Updates (Taller Plot, Wider Margins)
    fig.update_layout(
        height=max(450 * n_kpis, 1000),  # Adjusted height
        title_text=f"<b>{title if title else 'Raw KPIs and Influence Scores'}</b>",
        title_font=dict(size=24, family="Arial", color="black"),
        title_x=0.5,
        legend_title="<b>File Name and Data Type</b>",
        hovermode='x unified',
        font=dict(size=16, family="Arial"),  # Larger base font
        margin=dict(l=100, r=50, b=100, t=100)  # Wider left margin
    )

    base_filename = f"kpi_raw_and_is_improved_{dataset_name}"  # Simplified filename
    if specific_name:
        base_filename += f"_{specific_name}"

    png_filename = os.path.join(folder_path, f"{base_filename}.png")
    fig.write_image(png_filename)
    print(f"Plot saved as PNG: {png_filename}")

    html_filename = os.path.join(folder_path, f"{base_filename}.html")
    fig.write_html(html_filename)
    print(f"Plot saved as HTML: {html_filename}")

    fig.show()

# Example Usage:
kpi_columns = [col for col in IS_df_binary.columns if col not in ['Timestep', 'File_name', 'delta_type']]

plot_kpis_raw_and_multiple_is_improved(
    raw_df=raw_df.copy(),
    is_dfs=[IS_df_hierarchical.copy(), IS_df_decaying.copy()],
    is_styles=['dash', 'dot'],
    timestep_col='Timestep',
    file_name_col='File_name',
    kpis=kpi_columns,
    dataset_name=dataset,
    specific_name="hierarchical_vs_decaying_raw",  # Descriptive name
    title="Raw KPIs vs. Hierarchical and Decaying Influence Scores"  # Clear title
)

### Visualize with symbolic values

In [ ]:
def plot_kpis_symbolic_and_is_improved(symbolic_df, is_dfs, is_styles, timestep_col, file_name_col, kpis, dataset_name, specific_name="", title=None):
    """
    Plots KPIs and IS, reducing vertical space between subplots.
    """
    vis_name = "kpi_symbolic_and_multiple_is_improved"
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)

    n_kpis = len(kpis)
    # Reduce vertical_spacing
    fig = make_subplots(rows=n_kpis, cols=1,
                        subplot_titles=[f"<b>{kpi} (Symbolic Y-Axis)</b>" for kpi in kpis],
                        shared_xaxes=True, vertical_spacing=0.02,  # Reduced spacing
                        specs=[[{'secondary_y': True}] for _ in range(n_kpis)])

    file_names = natsorted(symbolic_df[file_name_col].unique())
    colors = qualitative.Plotly
    color_mapping = {file_name: colors[i % len(colors)] for i, file_name in enumerate(file_names)}

    for i, kpi in enumerate(kpis):
        unique_symbolic_values = symbolic_df[kpi].unique()
        sorted_symbolic_values = sort_symbolic_values(unique_symbolic_values)
        value_to_position = {value: pos for pos, value in enumerate(sorted_symbolic_values)}

        num_unique = len(sorted_symbolic_values)
        max_ticks = 10
        tickvals = []
        ticktext = []

        if num_unique <= max_ticks:
            tickvals = list(range(num_unique))
            ticktext = sorted_symbolic_values
        else:
            step = max(1, (num_unique - 1) // (max_ticks - 1))
            indices = list(range(0, num_unique, step))
            if indices[-1] != num_unique - 1:
                indices[-1] = num_unique - 1
            tickvals = indices
            ticktext = [sorted_symbolic_values[idx] for idx in indices]

        for j, file_name in enumerate(file_names):
            symbolic_df_filtered = symbolic_df[symbolic_df[file_name_col] == file_name]
            color = color_mapping[file_name]

            fig.add_trace(
                go.Scatter(
                    x=symbolic_df_filtered[timestep_col],
                    y=symbolic_df_filtered[kpi].map(value_to_position),
                    mode='lines',
                    name=f"{file_name} (Symbolic)",
                    line=dict(color=color),
                    legendgroup=file_name,
                    showlegend=(i == 0)
                ),
                row=i + 1, col=1, secondary_y=False
            )

            for is_df, style in zip(is_dfs, is_styles):
                is_df_type = is_df['delta_type'].iloc[0]
                is_df_filtered = is_df[is_df[file_name_col] == file_name]
                fig.add_trace(
                    go.Scatter(
                        x=is_df_filtered[timestep_col],
                        y=is_df_filtered[kpi],
                        mode='lines',
                        name=f"{file_name} (IS - {is_df_type}, Style: {style})",
                        line=dict(color=color, dash=style),
                        legendgroup=file_name,
                        showlegend=(i == 0)
                    ),
                    row=i + 1, col=1, secondary_y=True
                )

        fig.update_yaxes(
            title_text="<b>Symbolic Value</b>",
            secondary_y=False,
            row=i + 1, col=1,
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            tickangle=0,
            title_font=dict(size=16),
            tickfont=dict(size=14),
            automargin=True,
        )
        fig.update_yaxes(title_text="<b>Influence Score</b>", secondary_y=True, row=i + 1, col=1, title_font=dict(size=16), tickfont=dict(size=14))

    # Adjust overall height calculation
    fig.update_layout(
        height=max(450 * n_kpis, 1000),  # Adjusted height calculation
        title_text=f"<b>{title if title else 'Symbolic KPIs and Influence Scores'}</b>",
        title_font=dict(size=24, family="Arial", color="black"),
        title_x=0.5,
        legend_title="<b>File Name and Data Type</b>",
        hovermode='x unified',
        font=dict(size=16, family="Arial"),
        margin=dict(l=250, r=50, b=100, t=100)
    )
    base_filename = f"kpi_symbolic_and_is_improved_{dataset_name}"
    if specific_name:
        base_filename += f"_{specific_name}"

    png_filename = os.path.join(folder_path, f"{base_filename}.png")
    fig.write_image(png_filename)
    print(f"Plot saved as PNG: {png_filename}")

    html_filename = os.path.join(folder_path, f"{base_filename}.html")
    fig.write_html(html_filename)
    print(f"Plot saved as HTML: {html_filename}")

    fig.show()

# Example Usage (same as before, but with the new function):
kpi_columns = [col for col in IS_df_binary.columns if col not in ['Timestep', 'File_name', 'delta_type']]

plot_kpis_symbolic_and_is_improved(
    symbolic_df=symbolic_df.copy(),  # Replace with your symbolic DataFrame
    is_dfs=[IS_df_hierarchical.copy(), IS_df_decaying.copy()],  # Replace with your IS DataFrames
    is_styles=['dash', 'dot'],
    timestep_col='Timestep',
    file_name_col='File_name',
    kpis=kpi_columns,  # Replace with your KPI columns
    dataset_name=dataset,
    specific_name="hierarchical_vs_decaying_improved_compact",  # Descriptive name
    title="Symbolic KPIs vs. Hierarchical and Decaying Influence Scores (Improved Y-Axis)"
)

### Visualize using heatmap for symbolic without softmax

In [ ]:
def sort_symbolic_kpi_strings_key(symbolic_string):
    """
    Key function for sorting symbolic KPI strings.
    """
    level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
    parts = symbolic_string.split(", ")
    if len(parts) == 3:
        category = parts[1].strip()
        for level in level_order:
            if level in category:
                return level_order.index(level)
    else:
        for level in level_order:
            if level in symbolic_string:
                return level_order.index(level)
    return len(level_order)

def get_all_possible_symbolic_values(kpi_name):
    """
    Generates all possible symbolic values for a given KPI.
    """
    predicates = ['inc', 'const', 'dec']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    if kpi_name.endswith('_all'):
        extra_states = ['dropping', 'spiking', 'fluctuating']
        return [f"{p}({kpi_name}, {e})" for p in predicates for e in extra_states]
    else:
        return [f"{p}({kpi_name}, {c})" for p in predicates for c in categories]

def plot_all_heatmaps_in_one(symbolic_df, IS_df_binary, kpi_columns, dataset_name, specific_name="", bins=np.arange(0, 1.4, 0.1)):
    """
    Generates all frequency heatmaps as subplots in a single figure and saves it.
    """
    n_kpis = len(kpi_columns)
    n_cols = 3  # Number of columns for subplots (adjust as needed)
    n_rows = (n_kpis + n_cols - 1) // n_cols  # Calculate rows needed

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))  # Adjust figsize
    axes = axes.flatten()  # Flatten the axes array for easy iteration

    for i, kpi in enumerate(kpi_columns):
        if kpi not in symbolic_df.columns or kpi not in IS_df_binary.columns:
            print(f"Warning: KPI '{kpi}' not found in both DataFrames. Skipping.")
            continue

        # Get unique and sorted symbolic values
        unique_symbolic_values = symbolic_df[kpi].unique()
        sorted_unique_symbolic_values = sorted(unique_symbolic_values, key=sort_symbolic_kpi_strings_key, reverse=True)

        # Bin continuous values
        IS_df_binary_binned = IS_df_binary.copy()
        IS_df_binary_binned[kpi] = pd.cut(IS_df_binary[kpi], bins=bins, labels=[f"{bins[i]:.1f}" for i in range(len(bins)-1)], right=False)
        IS_df_binary_binned = IS_df_binary_binned.dropna(subset=[kpi])
        valid_indices = IS_df_binary_binned.index
        symbolic_df_filtered = symbolic_df.loc[valid_indices]
        IS_df_binary_binned = IS_df_binary_binned.loc[valid_indices]

        # Compute frequency counts
        freq_table = pd.crosstab(symbolic_df_filtered[kpi], IS_df_binary_binned[kpi])

        # Reindex with sorted unique symbolic values and bins
        freq_table = freq_table.reindex(index=sorted_unique_symbolic_values, columns=[f"{bins[i]:.1f}" for i in range(len(bins)-1)], fill_value=0)

        # Create annotation mask
        annot_mask = freq_table.values > 0

        # Plot the heatmap on the appropriate subplot
        sns.heatmap(freq_table, annot=np.where(annot_mask, freq_table.values, ''), fmt="", cmap="viridis", cbar_kws={'label': 'Count'}, annot_kws={"size": 8}, ax=axes[i])
        axes[i].set_title(f"{kpi}")  # Set subplot title
        axes[i].set_xlabel("Binned IS")
        axes[i].set_ylabel("Symbolic State")

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    
    # Save the figure
    vis_name = "IS_Score_heatmaps"
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
    base_filename = f"all_heatmaps_{dataset_name}"
    if specific_name:
        base_filename += f"_{specific_name}"
    png_filename = os.path.join(folder_path, f"{base_filename}.png")
    plt.savefig(png_filename)
    print(f"Heatmap saved as: {png_filename}")

    plt.show()


# Example Usage (assuming you have symbolic_df, IS_df_binary, kpi_columns, and dataset defined)

kpi_columns = [col for col in IS_df_binary.columns if col not in ['Timestep', 'File_name', 'delta_type']]  # Make sure this is defined!

plot_all_heatmaps_in_one(symbolic_df, IS_df_binary, kpi_columns, dataset_name=dataset)

### Visualize using heatmap for symbolic with softmax

In [ ]:
def softmax(x):
    """Compute softmax values for each sets of scores in x."""
    e_x = np.exp(x - np.max(x))  # Subtract max for numerical stability
    return e_x / e_x.sum()

def create_softmax_dataframe(df, kpi_columns):
    """
    Creates a new DataFrame with softmax values applied to KPI columns at each timestep.

    Args:
        df: The original DataFrame.
        kpi_columns: A list of column names representing the KPIs.

    Returns:
        A new DataFrame with the softmaxed values, along with original 'Timestep', 'File_name', and 'delta_type'.
        Returns None if there is any error.
    """
    # Input validation and error handling
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if not isinstance(kpi_columns, list):
        raise TypeError("kpi_columns must be a list")
    for col in kpi_columns:
        if col not in df.columns:
            raise ValueError(f"KPI column '{col}' not found in DataFrame")
    if not all(pd.api.types.is_numeric_dtype(df[col]) for col in kpi_columns):
        raise ValueError("All KPI columns must be numeric")
    if 'Timestep' not in df.columns or 'File_name' not in df.columns or 'delta_type' not in df.columns:
        raise ValueError("'Timestep', 'File_name', and 'delta_type' columns must be present in DataFrame")
    

    # Initialize lists to store the results.  We build lists of dictionaries,
    # then create the dataframe at the end, which is much more efficient than
    # appending to a dataframe row by row.
    softmax_data = []
    
    # Iterate through unique combinations of 'File_name' and 'delta_type'.
    for (file_name, delta_type), group in df.groupby(['File_name', 'delta_type']):
      for timestep, time_group in group.groupby('Timestep'):
        # Get the KPI values for the current timestep
        kpi_values = time_group[kpi_columns].values.flatten()

        # Apply softmax
        softmax_values = softmax(kpi_values)

        # Create a dictionary for the current row, preserving Timestep, File_name, and delta_type
        row_dict = {
            'Timestep': timestep,
            'File_name': file_name,
            'delta_type': delta_type
        }
        # Add the softmax values to the dictionary, using the original KPI column names
        row_dict.update({kpi_col: softmax_val for kpi_col, softmax_val in zip(kpi_columns, softmax_values)})
        softmax_data.append(row_dict)

    # Create the new DataFrame
    softmax_df = pd.DataFrame(softmax_data)
    return softmax_df


# Example usage (assuming IS_df_binary and kpi_columns are defined as before)
kpi_columns = [col for col in IS_df_binary.columns if col not in ['Timestep', 'File_name', 'delta_type']]
softmax_df = create_softmax_dataframe(IS_df_binary, kpi_columns)

def sort_symbolic_kpi_strings_key(symbolic_string):
    """
    Key function for sorting symbolic KPI strings.
    """
    level_order = ["VeryHigh", "High", "Medium", "Low", "VeryLow"]
    parts = symbolic_string.split(", ")
    if len(parts) == 3:
        category = parts[1].strip()
        for level in level_order:
            if level in category:
                return level_order.index(level)
    else:
        for level in level_order:
            if level in symbolic_string:
                return level_order.index(level)
    return len(level_order)

def get_all_possible_symbolic_values(kpi_name):
    """
    Generates all possible symbolic values for a given KPI.
    """
    predicates = ['inc', 'const', 'dec']
    categories = ['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh']
    if kpi_name.endswith('_all'):
        extra_states = ['dropping', 'spiking', 'fluctuating']
        return [f"{p}({kpi_name}, {e})" for p in predicates for e in extra_states]
    else:
        return [f"{p}({kpi_name}, {c})" for p in predicates for c in categories]

def plot_all_heatmaps_in_one(symbolic_df, IS_df_binary, kpi_columns, dataset_name, specific_name="", bins=np.arange(0, 0.4, 0.05)):
    """
    Generates all frequency heatmaps as subplots in a single figure and saves it.
    Uses bins from 0 to 0.4 with steps of 0.04 for the x-axis.
    """
    n_kpis = len(kpi_columns)
    n_cols = 3  # Number of columns for subplots (adjust as needed)
    n_rows = (n_kpis + n_cols - 1) // n_cols  # Calculate rows needed

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))  # Adjust figsize
    if n_kpis==1:
        axes = np.array([axes])
    axes = axes.flatten()  # Flatten the axes array for easy iteration


    for i, kpi in enumerate(kpi_columns):
        if kpi not in symbolic_df.columns or kpi not in IS_df_binary.columns:
            print(f"Warning: KPI '{kpi}' not found in both DataFrames. Skipping.")
            continue

        # Get unique and sorted symbolic values
        unique_symbolic_values = symbolic_df[kpi].unique()
        sorted_unique_symbolic_values = sorted(unique_symbolic_values, key=sort_symbolic_kpi_strings_key, reverse=True)

        # Bin continuous values
        IS_df_binary_binned = IS_df_binary.copy()
        # Use the provided bins (0 to 0.4 with steps of 0.04)
        IS_df_binary_binned[kpi] = pd.cut(IS_df_binary[kpi], bins=bins, labels=[f"{bins[i]:.2f}" for i in range(len(bins)-1)], right=False)
        IS_df_binary_binned = IS_df_binary_binned.dropna(subset=[kpi])
        valid_indices = IS_df_binary_binned.index
        symbolic_df_filtered = symbolic_df.loc[valid_indices]
        IS_df_binary_binned = IS_df_binary_binned.loc[valid_indices]


        # Compute frequency counts
        freq_table = pd.crosstab(symbolic_df_filtered[kpi], IS_df_binary_binned[kpi])

        # Reindex with sorted unique symbolic values and bins
        freq_table = freq_table.reindex(index=sorted_unique_symbolic_values, columns=[f"{bins[i]:.2f}" for i in range(len(bins)-1)], fill_value=0)

        # Create annotation mask
        annot_mask = freq_table.values > 0

        # Plot the heatmap on the appropriate subplot
        sns.heatmap(freq_table, annot=np.where(annot_mask, freq_table.values, ''), fmt="", cmap="viridis", cbar_kws={'label': 'Count'}, annot_kws={"size": 8}, ax=axes[i])
        axes[i].set_title(f"{kpi}")  # Set subplot title
        axes[i].set_xlabel("Binned IS (Softmax)")  # Updated x-axis label
        axes[i].set_ylabel("Symbolic State")

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    
    # Save the figure
    vis_name = "softmax_heatmaps"  # Changed folder name
    folder_path = create_visualization_folder_path(vis_name, dataset_name, specific_name)
    base_filename = f"all_heatmaps_{dataset_name}"
    if specific_name:
        base_filename += f"_{specific_name}"
    png_filename = os.path.join(folder_path, f"{base_filename}.png")
    plt.savefig(png_filename)
    print(f"Heatmap saved as: {png_filename}")

    plt.show()


# Example Usage (assuming you have symbolic_df, IS_df_binary (now softmax_df), kpi_columns, and dataset defined)

kpi_columns = [col for col in softmax_df.columns if col not in ['Timestep', 'File_name', 'delta_type']]

plot_all_heatmaps_in_one(symbolic_df, softmax_df, kpi_columns, dataset_name=dataset, specific_name="softmaxed")

In [ ]:
import plotly.express as px

# Melt the DataFrame so that each KPI's influence scores become a variable
df_melted = IS_df_binary.melt(
    id_vars=['Timestep', 'File_name', 'delta_type'],
    value_vars=kpi_columns,  # using previously defined kpi_columns
    var_name='KPI',
    value_name='Influence_Score'
)

fig = px.box(
    df_melted,
    x='KPI',
    y='Influence_Score',
    points="all",
    title="Distribution of Influence Scores per KPI (Binary)"
)
fig.show()